In [1]:
%%writefile config.py
"""Configuration for DINOv2 + MLP Copy-Move Forgery Detection"""
import torch

class Config:
    # ==================== PATHS ====================
    TRAIN_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/authentic"
    TRAIN_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/forged/images"
    TRAIN_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/masks"
    
    TEST_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_images/authentic"
    TEST_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_images/forged"
    TEST_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_masks"
    
    HQ_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection"
    HQ_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/supplemental_images"
    HQ_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/supplemental_masks"
    
    OUTPUT_DIR = "/kaggle/working/outputs"
    CHECKPOINT_DIR = "/kaggle/input/sci-forge-sim-arm/pytorch/default/1"
    
    # DINOv2 Model Path (Kaggle Dataset)
    DINOV2_PATH = "/kaggle/input/dinov2-pytorch"  # Adjust to your Kaggle dataset path
    
    # ==================== MODEL ====================
    INPUT_SIZE = 462  # 462/14 = 33 patches
    PATCH_SIZE = 14
    FEATURE_MAP_SIZE = 33  # 462 / 14
    EVAL_MASK_SIZE = 224  # Upsample predictions to 224 for evaluation
    
    DINO_EMBEDDING_DIM = 768  # ViT-B/14
    MLP_HIDDEN_DIM = 1024
    FINAL_EMBEDDING_DIM = 512
    
    MAX_INSTANCES = 5  # Maximum forgery instances (excluding background)
    
    # ==================== TRAINING ====================
    NUM_EPOCHS = 40
    BATCH_SIZE = 28  # DINOv2 is memory-intensive
    GRADIENT_ACCUMULATION_STEPS = 2
    
    ENCODER_FREEZE_EPOCHS = 10  # Freeze DINOv2 for first 10 epochs
    
    # Learning Rates
    MLP_LEARNING_RATE = 1e-3
    DINO_LEARNING_RATE = 1e-5  # Much lower to avoid forgetting
    WEIGHT_DECAY = 1e-4
    
    # Scheduler
    T_MAX = NUM_EPOCHS
    ETA_MIN = 1e-7
    
    # Mixed Precision
    USE_AMP = True
    
    # ==================== LOSS WEIGHTS ====================
    # PRIORITY: Separation >> Uniformity
    LOSS_WEIGHT_UNIFORMITY = 0.5    # Reduced - only penalize if variance too high
    LOSS_WEIGHT_SEPARATION = 2.5    # INCREASED - 5x more important than uniformity
    LOSS_WEIGHT_TRIPLET = 0.4       # Optional: Hard mining
    
    USE_TRIPLET = True  # Set to False to disable triplet loss entirely
    
    # Loss hyperparameters
    UNIFORMITY_THRESHOLD = 0.1      # NEW: Only penalize if variance > this threshold
    SEPARATION_MARGIN = 2.0         # Minimum distance between instance centroids
    TRIPLET_MARGIN = 0.5            # Triplet loss margin (if enabled)
    TRIPLET_HARD_MINING = True
    
    # Legacy (kept for compatibility, not used with new losses)
    LOSS_WEIGHT_INFONCE = 0.0
    INFONCE_TEMPERATURE = 0.07
    
    # ==================== AUGMENTATION ====================
    AUG_ROTATION_LIMIT = 30
    AUG_PROB_FLIP = 0.5
    AUG_PROB_ROTATE = 0.5
    
    AUG_BRIGHTNESS = 0.25
    AUG_CONTRAST = 0.15
    AUG_SATURATION = 0.15
    AUG_HUE = 0.05
    AUG_PROB_COLOR = 0.5
    
    AUG_JPEG_QUALITY_RANGE = (80, 100)
    AUG_PROB_JPEG = 0.4
    
    AUG_BLUR_LIMIT = (3, 5)
    AUG_PROB_BLUR = 0.3
    
    AUG_NOISE_VAR_LIMIT = (5.0, 15.0)
    AUG_PROB_NOISE = 0.3
    
    # ==================== INFERENCE (HDBSCAN) ====================
    HDBSCAN_MIN_CLUSTER_SIZE = 15  # Tunable
    HDBSCAN_MIN_SAMPLES = 5  # Tunable
    HDBSCAN_METRIC = 'euclidean'
    HDBSCAN_CLUSTER_SELECTION_METHOD = 'eom'  # Excess of Mass
    
    # NEW: Geometry validation threshold
    GEOMETRY_THRESHOLD = 0.5  # Cosine distance threshold - clusters must be this far from background
    
    # ==================== VALIDATION ====================
    VAL_SAMPLE_RATIO = 0.15  # Use 30% of batches (memory-efficient batched processing)
    VAL_AUTHENTIC_RATIO = 0.3  # 30% authentic
    VAL_FORGED_RATIO = 0.7  # 70% forged
    VAL_FREQUENCY = 3  # Validate every 3 epochs
    
    # Early Stopping
    EARLY_STOPPING_PATIENCE = 10  # Stop if no improvement for N validations
    EARLY_STOPPING_MIN_DELTA = 0.001  # Minimum improvement to count
    
    # ==================== THRESHOLD TUNING ====================
    TUNING_SAMPLE_RATIO = 0.20  # 20% of test set
    TUNING_AUTHENTIC_RATIO = 0.30
    TUNING_FORGED_RATIO = 0.70
    
    # HDBSCAN parameter grid - COARSE SEARCH FIRST
    TEST_MIN_CLUSTER_SIZES = [10, 15, 20, 25]
    TEST_MIN_SAMPLES = [3, 5, 7, 10]
    TEST_GEOMETRY_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]  # NEW: Geometry threshold search
    
    # Hierarchical search strategy
    USE_HIERARCHICAL_SEARCH = True  # Enable smart search to reduce combinations
    
    # ==================== DISTRIBUTED ====================
    SAVE_TOP_PREDICTIONS = 25  # Save 25 best, 25 moderate, 25 worst
    WORLD_SIZE = 2
    BACKEND = "nccl"
    
    # ==================== SMOKE TEST ====================
    SMOKE_TEST = False
    SMOKE_TEST_SAMPLES = 30
    SMOKE_TEST_EPOCHS = 15
    SMOKE_TEST_BATCH_SIZE = 4
    SMOKE_TEST_FREEZE_EPOCHS = 5
    SMOKE_TEST_GRAD_ACCUM = 1
    
    @classmethod
    def set_smoke_test(cls, enable=True):
        cls.SMOKE_TEST = enable
        if enable:
            cls.NUM_EPOCHS = cls.SMOKE_TEST_EPOCHS
            cls.ENCODER_FREEZE_EPOCHS = cls.SMOKE_TEST_FREEZE_EPOCHS
            cls.T_MAX = cls.SMOKE_TEST_EPOCHS
            cls.BATCH_SIZE = cls.SMOKE_TEST_BATCH_SIZE
            cls.GRADIENT_ACCUMULATION_STEPS = cls.SMOKE_TEST_GRAD_ACCUM
            print(f"🔥 SMOKE TEST MODE ENABLED")
    
    @classmethod
    def print_config(cls):
        print("=" * 60)
        print("DINOv2 + MLP COPY-MOVE FORGERY DETECTION")
        print("=" * 60)
        for key, value in cls.__dict__.items():
            if not key.startswith('_') and not callable(value):
                print(f"{key}: {value}")
        print("=" * 60)

Writing config.py


In [2]:
%%writefile utils.py
"""
Utility functions: RLE encoding/decoding and OF1 metric
Save as: utils.py
"""

import json
import numba
import numpy as np
from numba import types
import numpy.typing as npt
import pandas as pd
import scipy.optimize


class ParticipantVisibleError(Exception):
    pass


@numba.jit(nopython=True)
def _rle_encode_jit(x: npt.NDArray, fg_val: int = 1) -> list[int]:
    """Numba-jitted RLE encoder."""
    dots = np.where(x.T.flatten() == fg_val)[0]
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return run_lengths


def rle_encode(masks: list[npt.NDArray], fg_val: int = 1) -> str:
    """
    Encode masks to RLE format.
    Args:
        masks: list of numpy array of shape (height, width), 1 - mask, 0 - background
    Returns: run length encodings as a string, with each RLE JSON-encoded and separated by a semicolon.
    """
    return ';'.join([json.dumps(_rle_encode_jit(x, fg_val)) for x in masks])


@numba.njit
def _rle_decode_jit(mask_rle: npt.NDArray, height: int, width: int) -> npt.NDArray:
    """
    Decode RLE to mask.
    s: numpy array of run-length encoding pairs (start, length)
    shape: (height, width) of array to return
    Returns numpy array, 1 - mask, 0 - background
    """
    if len(mask_rle) % 2 != 0:
        raise ValueError('One or more rows has an odd number of values.')

    starts, lengths = mask_rle[0::2], mask_rle[1::2]
    starts -= 1
    ends = starts + lengths
    for i in range(len(starts) - 1):
        if ends[i] > starts[i + 1]:
            raise ValueError('Pixels must not be overlapping.')
    img = np.zeros(height * width, dtype=np.bool_)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img


def rle_decode(mask_rle: str, shape: tuple[int, int]) -> npt.NDArray:
    """
    Decode RLE string to mask.
    mask_rle: run-length as string formatted (start length)
              empty predictions need to be encoded with '-'
    shape: (height, width) of array to return
    Returns numpy array, 1 - mask, 0 - background
    """
    mask_rle = json.loads(mask_rle)
    mask_rle = np.asarray(mask_rle, dtype=np.int32)
    starts = mask_rle[0::2]
    if sorted(starts) != list(starts):
        raise ParticipantVisibleError('Submitted values must be in ascending order.')
    try:
        return _rle_decode_jit(mask_rle, shape[0], shape[1]).reshape(shape, order='F')
    except ValueError as e:
        raise ParticipantVisibleError(str(e)) from e


def calculate_f1_score(pred_mask: npt.NDArray, gt_mask: npt.NDArray):
    """Calculate F1 score between two binary masks."""
    pred_flat = pred_mask.flatten()
    gt_flat = gt_mask.flatten()

    tp = np.sum((pred_flat == 1) & (gt_flat == 1))
    fp = np.sum((pred_flat == 1) & (gt_flat == 0))
    fn = np.sum((pred_flat == 0) & (gt_flat == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    if (precision + recall) > 0:
        return 2 * (precision * recall) / (precision + recall)
    else:
        return 0


def calculate_f1_matrix(pred_masks: list[npt.NDArray], gt_masks: list[npt.NDArray]):
    """
    Calculate F1 matrix for all pairs of predicted and ground truth masks.
    """
    num_instances_pred = len(pred_masks)
    num_instances_gt = len(gt_masks)
    f1_matrix = np.zeros((num_instances_pred, num_instances_gt))

    for i in range(num_instances_pred):
        for j in range(num_instances_gt):
            pred_flat = pred_masks[i].flatten()
            gt_flat = gt_masks[j].flatten()
            f1_matrix[i, j] = calculate_f1_score(pred_mask=pred_flat, gt_mask=gt_flat)

    if f1_matrix.shape[0] < len(gt_masks):
        # Add rows of zeros if fewer predictions than ground truth
        f1_matrix = np.vstack((f1_matrix, np.zeros((len(gt_masks) - len(f1_matrix), num_instances_gt))))

    return f1_matrix


def oF1_score(pred_masks: list[npt.NDArray], gt_masks: list[npt.NDArray]):
    """
    Calculate optimal F1 score using Hungarian algorithm.
    """
    f1_matrix = calculate_f1_matrix(pred_masks, gt_masks)
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(-f1_matrix)
    excess_predictions_penalty = len(gt_masks) / max(len(pred_masks), len(gt_masks))
    return np.mean(f1_matrix[row_ind, col_ind]) * excess_predictions_penalty


def evaluate_single_image(label_rles: str, prediction_rles: str, shape_str: str) -> float:
    """Evaluate single image with RLE inputs."""
    shape = json.loads(shape_str)
    label_rles = [rle_decode(x, shape=shape) for x in label_rles.split(';')]
    prediction_rles = [rle_decode(x, shape=shape) for x in prediction_rles.split(';')]
    return oF1_score(prediction_rles, label_rles)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculate overall score for entire dataset.
    """
    df = solution
    df = df.rename(columns={'annotation': 'label'})

    df['prediction'] = submission['annotation']
    # Check for correct 'authentic' label
    authentic_indices = (df['label'] == 'authentic') | (df['prediction'] == 'authentic')
    df['image_score'] = ((df['label'] == df['prediction']) & authentic_indices).astype(float)

    df.loc[~authentic_indices, 'image_score'] = df.loc[~authentic_indices].apply(
        lambda row: evaluate_single_image(row['label'], row['prediction'], row['shape']), axis=1
    )
    return float(np.mean(df['image_score']))

Writing utils.py


In [3]:
%%writefile dataset.py
"""Dataset for DINOv2 Copy-Move Detection"""
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import albumentations as A
import cv2
import torch.nn.functional as F

class ForgeryDataset(Dataset):
    def __init__(self, authentic_dir, forged_dir, mask_dir, transform=None, config=None):
        self.authentic_dir = authentic_dir
        self.forged_dir = forged_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.config = config
        
        self.samples = []
        if os.path.exists(authentic_dir):
            auth_imgs = sorted([f for f in os.listdir(authentic_dir) 
                              if f.endswith(('.jpg', '.png', '.jpeg'))])
            for img in auth_imgs:
                self.samples.append({'image': img, 'is_forged': False, 'dir': authentic_dir})
                
        if os.path.exists(forged_dir):
            forg_imgs = sorted([f for f in os.listdir(forged_dir) 
                              if f.endswith(('.jpg', '.png', '.jpeg'))])
            for img in forg_imgs:
                self.samples.append({'image': img, 'is_forged': True, 'dir': forged_dir})
        
        print(f"📦 Total samples: {len(self.samples)}")
        
        if config and config.SMOKE_TEST:
            random.shuffle(self.samples)
            self.samples = self.samples[:config.SMOKE_TEST_SAMPLES]
            print(f"🔥 SMOKE TEST: Sampled {len(self.samples)} images")
    
    def __len__(self):
        return len(self.samples)
    
    def letterbox_resize(self, image, masks, target_size):
        """Letterbox resize maintaining aspect ratio"""
        h, w = image.shape[:2]
        scale = min(target_size / h, target_size / w)
        new_h, new_w = int(h * scale), int(w * scale)
        
        resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        padded_image = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        y_off = (target_size - new_h) // 2
        x_off = (target_size - new_w) // 2
        padded_image[y_off:y_off+new_h, x_off:x_off+new_w] = resized_image
        
        padded_masks = []
        for mask in masks:
            m_resized = cv2.resize(mask.astype(np.uint8), (new_w, new_h), 
                                  interpolation=cv2.INTER_NEAREST)
            p_mask = np.zeros((target_size, target_size), dtype=np.uint8)
            p_mask[y_off:y_off+new_h, x_off:x_off+new_w] = m_resized
            padded_masks.append(p_mask)
            
        return padded_image, padded_masks
    
    def downsample_masks_maxpool(self, masks, config):
        """
        Downsample masks: MaxPool(4x4) -> 144x144 -> Interpolate -> 33x33
        This preserves small forgeries better than direct interpolation
        """
        if not masks:
            return []
        
        # Stack masks: (K, H, W) where H=W=462
        masks_tensor = torch.from_numpy(np.stack(masks)).float().unsqueeze(1)  # (K, 1, 462, 462)
        
        # Step 1: MaxPool 4x4 -> 144x144 (462/4 ≈ 115, but we pad to 464 first)
        # Pad to 464 to make divisible by 4
        pad = (464 - 462) // 2
        if pad > 0:
            masks_tensor = F.pad(masks_tensor, (pad, pad, pad, pad), mode='constant', value=0)
        
        masks_tensor = F.max_pool2d(masks_tensor, kernel_size=4, stride=4)  # (K, 1, 116, 116)
        
        # Crop to 144x144 (we need to interpolate anyway, so let's go to 144)
        masks_tensor = F.interpolate(masks_tensor, size=(144, 144), 
                                     mode='nearest')  # (K, 1, 144, 144)
        
        # Step 2: Interpolate to 33x33
        masks_tensor = F.interpolate(masks_tensor, size=(config.FEATURE_MAP_SIZE, 
                                                         config.FEATURE_MAP_SIZE), 
                                     mode='bilinear', align_corners=False)  # (K, 1, 33, 33)
        
        # Threshold back to binary
        masks_tensor = (masks_tensor > 0.3).float().squeeze(1)  # (K, 33, 33)
        
        return [masks_tensor[i].numpy() for i in range(masks_tensor.shape[0])]
    
    def __getitem__(self, idx):
        try:
            sample = self.samples[idx]
            img_path = os.path.join(sample['dir'], sample['image'])
            
            image = cv2.imread(img_path)
            if image is None:
                raise ValueError("Image not found")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            masks = []
            if sample['is_forged']:
                mask_path = os.path.join(self.mask_dir, 
                                        os.path.splitext(sample['image'])[0] + '.npy')
                if os.path.exists(mask_path):
                    mask_data = np.load(mask_path, allow_pickle=True)
                    h, w = image.shape[:2]
                    
                    if mask_data.ndim == 2:
                        if mask_data.shape != (h, w):
                            mask_data = cv2.resize(mask_data.astype(np.uint8), (w, h), 
                                                  interpolation=cv2.INTER_NEAREST)
                        masks.append(mask_data)
                    elif mask_data.ndim == 3:
                        for i in range(mask_data.shape[0]):
                            if np.sum(mask_data[i]) > 0:
                                m = mask_data[i]
                                if m.shape != (h, w):
                                    m = cv2.resize(m.astype(np.uint8), (w, h), 
                                                  interpolation=cv2.INTER_NEAREST)
                                masks.append(m)
            
            # Letterbox to 462x462
            image, masks = self.letterbox_resize(image, masks, self.config.INPUT_SIZE)
            
            # Apply augmentation (before mask downsampling)
            if self.transform:
                if masks:
                    aug = self.transform(image=image, masks=masks)
                    image, masks = aug['image'], aug['masks']
                else:
                    image = self.transform(image=image)['image']
            
            # Downsample masks: 462x462 -> MaxPool -> 144 -> 33x33
            masks_33 = self.downsample_masks_maxpool(masks, self.config)
            
            # Pad to 6 instances (including background)
            while len(masks_33) < 6:
                masks_33.append(np.zeros((self.config.FEATURE_MAP_SIZE, 
                                         self.config.FEATURE_MAP_SIZE), dtype=np.float32))
            masks_33 = masks_33[:6]
            
            # Convert to tensors
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            masks_33 = torch.stack([torch.from_numpy(m.astype(np.float32)) for m in masks_33])
            
            return {
                'input': image,  # (3, 462, 462)
                'masks': masks_33,  # (6, 33, 33)
                'is_forged': torch.tensor(sample['is_forged'], dtype=torch.float32),
                'num_instances': torch.tensor(len([m for m in masks_33 if m.sum() > 0]), 
                                             dtype=torch.long),
                'image_name': sample['image']
            }
        except Exception as e:
            print(f"Error loading {sample['image']}: {e}")
            return self.__getitem__(random.randint(0, len(self.samples)-1))

def get_train_transform(config):
    return A.Compose([
        A.HorizontalFlip(p=config.AUG_PROB_FLIP),
        A.VerticalFlip(p=config.AUG_PROB_FLIP),
        A.Rotate(limit=config.AUG_ROTATION_LIMIT, p=config.AUG_PROB_ROTATE),
        A.GaussianBlur(blur_limit=config.AUG_BLUR_LIMIT, p=config.AUG_PROB_BLUR),
        A.GaussNoise(var_limit=config.AUG_NOISE_VAR_LIMIT, p=config.AUG_PROB_NOISE),
        A.ImageCompression(quality_range=config.AUG_JPEG_QUALITY_RANGE, 
                          p=config.AUG_PROB_JPEG),
        A.ColorJitter(p=config.AUG_PROB_COLOR)
    ])

def get_val_transform(config):
    return None

def create_dataloaders(config, rank=0, world_size=1):
    train_ds = ForgeryDataset(
        config.TRAIN_AUTHENTIC_DIR, 
        config.TRAIN_FORGED_DIR, 
        config.TRAIN_MASK_DIR, 
        get_train_transform(config), 
        config
    )
    
    val_ds = ForgeryDataset(
        config.TEST_AUTHENTIC_DIR, 
        config.TEST_FORGED_DIR, 
        config.TEST_MASK_DIR, 
        get_val_transform(config), 
        config
    )
    
    train_sampler = DistributedSampler(train_ds, num_replicas=world_size, 
                                       rank=rank, shuffle=True)
    val_sampler = DistributedSampler(val_ds, num_replicas=world_size, 
                                     rank=rank, shuffle=False)
    
    train_loader = DataLoader(
        train_ds, 
        batch_size=config.BATCH_SIZE, 
        sampler=train_sampler,
        num_workers=4, 
        pin_memory=True, 
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=config.BATCH_SIZE, 
        sampler=val_sampler,
        num_workers=4, 
        pin_memory=True
    )
    
    return train_loader, val_loader, train_sampler, val_sampler

Writing dataset.py


In [4]:
%%writefile model.py
"""DINOv2 + MLP Model for Copy-Move Forgery Detection"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys

class MLP(nn.Module):
    """3-layer MLP: 768 → 1024 → 768 → 512"""
    def __init__(self, input_dim=768, hidden_dim=1024, output_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, input_dim)
        self.fc3 = nn.Linear(input_dim, output_dim)
        self.relu = nn.ReLU(inplace=True)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(input_dim)
        
    def forward(self, x):
        # x: (B, N, 768) where N = 33*33 = 1089
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.relu(x)
        
        x = self.fc2(x)
        x = self.norm2(x)
        x = self.relu(x)
        
        x = self.fc3(x)  # (B, N, 512)
        
        # L2 normalize embeddings
        x = F.normalize(x, p=2, dim=-1)
        return x

class DINOv2ForgeryDetector(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Load DINOv2 ViT-B/14 from local path or torch hub
        self.dino = self._load_dinov2(config)
        
        # Freeze DINOv2 initially
        self.freeze_dino()
        
        # MLP projection head
        self.mlp = MLP(
            input_dim=config.DINO_EMBEDDING_DIM,
            hidden_dim=config.MLP_HIDDEN_DIM,
            output_dim=config.FINAL_EMBEDDING_DIM
        )
    
    def _load_dinov2(self, config):
        """Load DINOv2 model from local path or torch hub"""
        # Try loading from Kaggle dataset first
        if os.path.exists(config.DINOV2_PATH):
            print(f"📦 Loading DINOv2 from local path: {config.DINOV2_PATH}")
            try:
                # Add DINOv2 path to Python path
                dinov2_repo_path = config.DINOV2_PATH
                if dinov2_repo_path not in sys.path:
                    sys.path.insert(0, dinov2_repo_path)
                
                # Import and create model
                from dinov2.models.vision_transformer import vit_base
                
                model = vit_base(
                    patch_size=14,
                    img_size=518,
                    init_values=1.0,
                    block_chunks=0
                )
                
                # Load pretrained weights if available
                weights_path = os.path.join(dinov2_repo_path, 'dinov2_vitb14_pretrain.pth')
                if os.path.exists(weights_path):
                    print(f"📦 Loading pretrained weights from {weights_path}")
                    state_dict = torch.load(weights_path, map_location='cpu')
                    model.load_state_dict(state_dict, strict=False)
                else:
                    print("⚠️ No pretrained weights found, using random initialization")
                
                return model
                
            except Exception as e:
                print(f"⚠️ Failed to load from local path: {e}")
                print("📥 Falling back to torch.hub download...")
        
        # Fallback: Download from torch hub (only on rank 0)
        print("📥 Downloading DINOv2 from torch.hub...")
        import torch.distributed as dist
        
        # Only rank 0 downloads, others wait
        if dist.is_initialized():
            rank = dist.get_rank()
            if rank == 0:
                model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', trust_repo=True)
            dist.barrier()  # Wait for rank 0 to finish download
            if rank != 0:
                model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', trust_repo=True)
        else:
            model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', trust_repo=True)
        
        return model
        
    def freeze_dino(self):
        for param in self.dino.parameters():
            param.requires_grad = False
        print("🧊 DINOv2 encoder frozen")
    
    def unfreeze_dino(self):
        for param in self.dino.parameters():
            param.requires_grad = True
        print("🔥 DINOv2 encoder unfrozen")
    
    def forward(self, x):
        """
        Args:
            x: (B, 3, 462, 462)
        Returns:
            embeddings: (B, 33, 33, 512) - patch embeddings
        """
        B = x.shape[0]
        
        # DINOv2 forward (outputs patch tokens, no CLS)
        with torch.cuda.amp.autocast(enabled=self.config.USE_AMP):
            # Get patch features
            features = self.dino.forward_features(x)
            patch_tokens = features['x_norm_patchtokens']  # (B, N, 768) where N=33*33
        
        # Pass through MLP
        embeddings = self.mlp(patch_tokens)  # (B, N, 512)
        
        # Reshape to spatial grid
        N = patch_tokens.shape[1]
        H = W = int(N ** 0.5)  # Should be 33
        embeddings = embeddings.reshape(B, H, W, -1)  # (B, 33, 33, 512)
        
        return embeddings
    
    def get_learnable_params(self):
        """Return MLP and DINOv2 params separately for different LRs"""
        mlp_params = list(self.mlp.parameters())
        dino_params = list(self.dino.parameters())
        return mlp_params, dino_params

def create_model(config):
    return DINOv2ForgeryDetector(config)

Writing model.py


In [5]:
%%writefile clustering.py
"""HDBSCAN Clustering for Inference with Candidate Validation"""
import numpy as np
import torch
import torch.nn.functional as F
from hdbscan import HDBSCAN
import cv2

def cluster_embeddings(embeddings, config):
    """
    Cluster 33x33 embeddings using HDBSCAN
    Args:
        embeddings: (33, 33, 512) numpy array
        config: Config object
    Returns:
        cluster_labels: (33, 33) numpy array, -1 for noise, 0+ for clusters
    """
    H, W, D = embeddings.shape
    emb_flat = embeddings.reshape(-1, D)  # (N, 512)
    
    # Run HDBSCAN with all CPU cores
    clusterer = HDBSCAN(
        min_cluster_size=config.HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=config.HDBSCAN_MIN_SAMPLES,
        metric=config.HDBSCAN_METRIC,
        cluster_selection_method=config.HDBSCAN_CLUSTER_SELECTION_METHOD,
        core_dist_n_jobs=-1  # Use all available CPU cores
    )
    
    cluster_labels = clusterer.fit_predict(emb_flat)  # (N,)
    cluster_labels = cluster_labels.reshape(H, W)  # (33, 33)
    
    return cluster_labels

def validate_candidates(cluster_labels, embeddings, config):
    """
    Validate candidate forgery clusters by measuring their distance from background
    
    Args:
        cluster_labels: (33, 33) numpy array
        embeddings: (33, 33, 512) numpy array
        config: Config object
    
    Returns:
        valid_cluster_ids: List of cluster IDs that are sufficiently far from background
    """
    H, W, D = embeddings.shape
    emb_flat = embeddings.reshape(-1, D)  # (N, 512)
    label_flat = cluster_labels.flatten()  # (N,)
    
    # Get unique clusters (exclude -1 noise)
    unique_clusters = np.unique(label_flat)
    unique_clusters = unique_clusters[unique_clusters >= 0]
    
    if len(unique_clusters) == 0:
        return []  # No clusters found
    
    # Count cluster sizes to identify background (largest cluster)
    cluster_sizes = [(c, (label_flat == c).sum()) for c in unique_clusters]
    cluster_sizes.sort(key=lambda x: x[1], reverse=True)
    
    background_cluster = cluster_sizes[0][0]
    
    # Compute background centroid
    bg_mask = (label_flat == background_cluster)
    bg_embeddings = emb_flat[bg_mask]
    
    if bg_embeddings.shape[0] == 0:
        return []  # No background found (shouldn't happen)
    
    bg_centroid = bg_embeddings.mean(axis=0)  # (512,)
    
    # Validate each candidate cluster
    valid_clusters = []
    
    for cluster_id, size in cluster_sizes[1:]:  # Skip background
        # Compute candidate centroid
        cand_mask = (label_flat == cluster_id)
        cand_embeddings = emb_flat[cand_mask]
        
        if cand_embeddings.shape[0] == 0:
            continue
        
        cand_centroid = cand_embeddings.mean(axis=0)  # (512,)
        
        # Compute cosine distance
        # Cosine distance = 1 - cosine_similarity
        bg_norm = bg_centroid / (np.linalg.norm(bg_centroid) + 1e-8)
        cand_norm = cand_centroid / (np.linalg.norm(cand_centroid) + 1e-8)
        cosine_sim = np.dot(bg_norm, cand_norm)
        cosine_dist = 1.0 - cosine_sim
        
        # Validate: distance must be greater than threshold
        if cosine_dist > config.GEOMETRY_THRESHOLD:
            valid_clusters.append(cluster_id)
    
    return valid_clusters

def generate_masks_from_clusters(cluster_labels, config, max_instances=5, embeddings=None):
    """
    Generate instance masks from cluster labels with candidate validation
    
    Args:
        cluster_labels: (33, 33) numpy array
        config: Config object
        max_instances: Maximum forgery instances (excluding background)
        embeddings: (33, 33, 512) numpy array - REQUIRED for validation
    
    Returns:
        masks: List of (224, 224) binary masks, upsampled to 224 for evaluation
    """
    H, W = cluster_labels.shape  # 33, 33
    
    # Get unique clusters (exclude -1 noise)
    unique_clusters = np.unique(cluster_labels)
    unique_clusters = unique_clusters[unique_clusters >= 0]
    
    if len(unique_clusters) == 0:
        return []  # No clusters found
    
    # Count cluster sizes
    cluster_sizes = [(c, (cluster_labels == c).sum()) for c in unique_clusters]
    cluster_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Largest cluster is background
    bg_cluster = cluster_sizes[0][0]
    
    # Get candidate clusters (all non-background)
    candidate_clusters = [c for c, _ in cluster_sizes[1:]]
    
    # VALIDATE CANDIDATES: Check if they're geometrically distinct from background
    if embeddings is not None and len(candidate_clusters) > 0:
        valid_clusters = validate_candidates(cluster_labels, embeddings, config)
        forgery_clusters = valid_clusters
    else:
        # Fallback: use all candidates (old behavior)
        forgery_clusters = candidate_clusters
    
    # If more than max_instances, keep largest ones
    if len(forgery_clusters) > max_instances:
        # Get sizes of valid clusters
        valid_sizes = [(c, (cluster_labels == c).sum()) for c in forgery_clusters]
        valid_sizes.sort(key=lambda x: x[1], reverse=True)
        forgery_clusters = [c for c, _ in valid_sizes[:max_instances]]
    
    # Generate masks (at 33x33 resolution first)
    masks_33 = []
    for cluster_id in forgery_clusters:
        mask = (cluster_labels == cluster_id).astype(np.float32)  # (33, 33)
        masks_33.append(mask)
    
    if len(masks_33) == 0:
        return []  # No valid forgeries found -> AUTHENTIC
    
    # Upsample to 224x224 for evaluation
    masks_224 = []
    for mask in masks_33:
        # Use patch-based upsampling (cover 14x14 patches)
        mask_462 = upsample_mask_patchwise(mask, patch_size=14, 
                                           target_size=462)  # (462, 462)
        # Downsample to 224 for evaluation
        mask_224 = cv2.resize(mask_462, (config.EVAL_MASK_SIZE, config.EVAL_MASK_SIZE), 
                             interpolation=cv2.INTER_NEAREST)
        masks_224.append(mask_224)
    
    return masks_224

def upsample_mask_patchwise(mask_33, patch_size=14, target_size=462):
    """
    Upsample 33x33 mask to target_size by covering patch_size x patch_size regions
    Args:
        mask_33: (33, 33) binary mask
        patch_size: Patch size (14 for DINOv2 ViT-B/14)
        target_size: Target image size (462)
    Returns:
        mask_target: (target_size, target_size) binary mask
    """
    H, W = mask_33.shape
    mask_target = np.zeros((target_size, target_size), dtype=np.float32)
    
    for i in range(H):
        for j in range(W):
            if mask_33[i, j] > 0.5:
                # Cover the corresponding 14x14 patch
                y_start = i * patch_size
                y_end = min((i + 1) * patch_size, target_size)
                x_start = j * patch_size
                x_end = min((j + 1) * patch_size, target_size)
                
                mask_target[y_start:y_end, x_start:x_end] = 1.0
    
    return mask_target

def predict_masks(model, image, config):
    """
    Predict forgery masks for a single image
    Args:
        model: DINOv2ForgeryDetector
        image: (3, 462, 462) tensor
        config: Config object
    Returns:
        masks: List of (224, 224) binary numpy masks
    """
    device = image.device
    model.eval()
    
    with torch.no_grad():
        # Get embeddings
        embeddings = model(image.unsqueeze(0))  # (1, 33, 33, 512)
        embeddings = embeddings[0].cpu().numpy()  # (33, 33, 512)
    
    # Cluster embeddings
    cluster_labels = cluster_embeddings(embeddings, config)
    
    # Generate masks WITH VALIDATION
    masks = generate_masks_from_clusters(cluster_labels, config, 
                                         max_instances=config.MAX_INSTANCES,
                                         embeddings=embeddings)  # Pass embeddings for validation
    
    return masks

Writing clustering.py


In [6]:
%%writefile losses.py
"""Uniformity + Inter-Instance Separation for Copy-Move Forgery Detection"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class UniformityLoss(nn.Module):
    """Minimize variance within instances (including background) - RELAXED VERSION"""
    def __init__(self, threshold=0.1):
        super().__init__()
        self.threshold = threshold  # Only penalize if variance > threshold
    
    def forward(self, embeddings, masks, is_forged):
        """
        Args:
            embeddings: (B, 33, 33, 512)
            masks: (B, 6, 33, 33) - ground truth masks at 33x33
            is_forged: (B,) - binary labels
        Returns:
            uniformity_loss: scalar
        """
        B, H, W, D = embeddings.shape
        device = embeddings.device
        
        total_loss = 0.0
        valid_batches = 0
        
        for b in range(B):
            emb = embeddings[b]  # (33, 33, 512)
            emb_flat = emb.reshape(-1, D)  # (N, 512) where N = 33*33
            
            if is_forged[b] < 0.5:
                # ========== AUTHENTIC IMAGE ==========
                # All patches should have low variance (uniform)
                mean_emb = emb_flat.mean(dim=0)  # (512,)
                variance = ((emb_flat - mean_emb) ** 2).sum(dim=1).mean()
                
                # RELAXED: Only penalize if variance exceeds threshold
                if variance > self.threshold:
                    total_loss += F.relu(variance - self.threshold)
                    valid_batches += 1
            
            else:
                # ========== FORGED IMAGE ==========
                # Create instance label map (0 = background, 1+ = forgery instances)
                curr_masks = masks[b]  # (6, 33, 33)
                label_map = torch.zeros((H, W), device=device, dtype=torch.long)
                
                active_instances = []
                for i in range(curr_masks.shape[0]):
                    mask_i = curr_masks[i]
                    if mask_i.sum() > 0:
                        label_map[mask_i > 0.5] = (i + 1)  # 1-indexed (0 is background)
                        active_instances.append(i + 1)
                
                label_flat = label_map.flatten()  # (N,)
                
                # Get all unique instances (including background = 0)
                unique_labels = torch.unique(label_flat)
                
                if len(unique_labels) == 0:
                    continue
                
                # Compute intra-instance variance for each instance
                instance_loss = 0.0
                instance_count = 0
                
                for label in unique_labels:
                    mask_idx = (label_flat == label)
                    instance_emb = emb_flat[mask_idx]  # (n_patches, 512)
                    
                    if instance_emb.shape[0] < 2:
                        continue  # Need at least 2 patches
                    
                    # Minimize variance within this instance
                    mean_inst = instance_emb.mean(dim=0)  # (512,)
                    variance = ((instance_emb - mean_inst) ** 2).sum(dim=1).mean()
                    
                    # RELAXED: Only penalize if variance exceeds threshold
                    if variance > self.threshold:
                        instance_loss += F.relu(variance - self.threshold)
                        instance_count += 1
                
                if instance_count > 0:
                    total_loss += instance_loss / instance_count
                    valid_batches += 1
        
        return total_loss / valid_batches if valid_batches > 0 else 0.0 * embeddings.sum()


class InterInstanceSeparationLoss(nn.Module):
    """Maximize distance between instance centroids"""
    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, embeddings, masks):
        """
        Args:
            embeddings: (B, 33, 33, 512)
            masks: (B, 6, 33, 33)
        Returns:
            separation_loss: scalar
        """
        B, H, W, D = embeddings.shape
        device = embeddings.device
        
        total_loss = 0.0
        valid_batches = 0
        
        for b in range(B):
            emb = embeddings[b].reshape(-1, D)  # (N, 512)
            curr_masks = masks[b]  # (6, 33, 33)
            
            # Build label map
            label_map = torch.zeros((H, W), device=device, dtype=torch.long)
            active_instances = []
            
            for i in range(curr_masks.shape[0]):
                mask_i = curr_masks[i]
                if mask_i.sum() > 0:
                    label_map[mask_i > 0.5] = (i + 1)
                    active_instances.append(i + 1)
            
            label_flat = label_map.flatten()
            unique_labels = torch.unique(label_flat)
            
            if len(unique_labels) < 2:
                continue  # Need at least 2 instances to separate
            
            # Compute centroids for each instance
            centroids = []
            centroid_labels = []
            
            for label in unique_labels:
                mask_idx = (label_flat == label)
                instance_emb = emb[mask_idx]
                
                if instance_emb.shape[0] > 0:
                    centroid = instance_emb.mean(dim=0)  # (512,)
                    centroids.append(centroid)
                    centroid_labels.append(label)
            
            if len(centroids) < 2:
                continue
            
            centroids = torch.stack(centroids)  # (K, 512)
            
            # Compute pairwise distances between centroids
            dist_matrix = torch.cdist(centroids, centroids, p=2)  # (K, K)
            
            # Loss: Encourage distances to be >= margin
            # We use hinge loss: max(0, margin - distance)
            loss = 0.0
            count = 0
            
            for i in range(len(centroids)):
                for j in range(i + 1, len(centroids)):
                    distance = dist_matrix[i, j]
                    # Penalize if distance < margin
                    loss += F.relu(self.margin - distance)
                    count += 1
            
            if count > 0:
                total_loss += loss / count
                valid_batches += 1
        
        return total_loss / valid_batches if valid_batches > 0 else 0.0 * embeddings.sum()


class TripletLossWithHardMining(nn.Module):
    """Original triplet loss (kept for compatibility, but can be replaced)"""
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin
    
    def forward(self, embeddings, masks):
        """
        Args:
            embeddings: (B, 33, 33, 512)
            masks: (B, 6, 33, 33) - ground truth masks at 33x33
        """
        B, H, W, D = embeddings.shape
        device = embeddings.device
        
        total_loss = 0.0
        valid_batches = 0
        
        for b in range(B):
            emb = embeddings[b]  # (33, 33, 512)
            emb_flat = emb.reshape(-1, D)  # (N, 512) where N = 33*33
            
            curr_masks = masks[b]  # (6, 33, 33)
            
            # Create instance label map
            label_map = torch.zeros((H, W), device=device, dtype=torch.long)
            active_ids = []
            
            for i in range(curr_masks.shape[0]):
                mask_i = curr_masks[i]
                if mask_i.sum() > 0:
                    label_map[mask_i > 0.5] = (i + 1)  # 1-indexed (0 is background)
                    active_ids.append(i + 1)
            
            label_flat = label_map.flatten()  # (N,)
            
            if len(active_ids) == 0:
                continue  # No forgery instances, skip
            
            # Compute pairwise distances
            dist_matrix = torch.cdist(emb_flat, emb_flat, p=2)  # (N, N)
            
            # Create masks for positive/negative pairs
            same_instance = (label_flat.unsqueeze(0) == label_flat.unsqueeze(1))
            same_instance.fill_diagonal_(False)  # Exclude self-pairs
            
            # Separate background-foreground and foreground-foreground negatives
            is_bg = (label_flat == 0)
            is_fg = ~is_bg
            
            # Positive pairs: same instance
            if same_instance.sum() == 0:
                continue
            
            # Hard positive mining (furthest positive)
            pos_dists = dist_matrix.clone()
            pos_dists[~same_instance] = -1e9  # Mask out non-positives
            hard_pos_dist, _ = pos_dists.max(dim=1)  # (N,)
            hard_pos_dist = hard_pos_dist[hard_pos_dist > -1e8]  # Valid positives
            
            if hard_pos_dist.numel() == 0:
                continue
            
            # Hard negative mining (closest negative)
            # Background-foreground negatives
            bg_fg_neg = is_bg.unsqueeze(0) ^ is_fg.unsqueeze(1)
            
            # Foreground-foreground negatives (different instances)
            fg_fg_neg = (~same_instance) & (is_fg.unsqueeze(0) & is_fg.unsqueeze(1))
            
            all_neg = bg_fg_neg | fg_fg_neg
            
            neg_dists = dist_matrix.clone()
            neg_dists[~all_neg] = 1e9  # Mask out non-negatives
            hard_neg_dist, _ = neg_dists.min(dim=1)  # (N,)
            hard_neg_dist = hard_neg_dist[hard_neg_dist < 1e8]  # Valid negatives
            
            if hard_neg_dist.numel() == 0:
                continue
            
            # Triplet loss: max(0, d_pos - d_neg + margin)
            # We need paired positives and negatives
            min_len = min(hard_pos_dist.shape[0], hard_neg_dist.shape[0])
            if min_len == 0:
                continue
            
            loss = F.relu(hard_pos_dist[:min_len] - hard_neg_dist[:min_len] + self.margin)
            total_loss += loss.mean()
            valid_batches += 1
        
        return total_loss / valid_batches if valid_batches > 0 else 0.0 * embeddings.sum()


class CombinedLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # New losses with RELAXED uniformity
        self.uniformity_loss = UniformityLoss(threshold=config.UNIFORMITY_THRESHOLD)
        self.separation_loss = InterInstanceSeparationLoss(margin=config.SEPARATION_MARGIN)
        
        # Keep triplet for backwards compatibility (can be disabled)
        self.triplet_loss = TripletLossWithHardMining(margin=config.TRIPLET_MARGIN)
    
    def forward(self, embeddings, masks, is_forged):
        """
        Args:
            embeddings: (B, 33, 33, 512)
            masks: (B, 6, 33, 33) - ground truth masks
            is_forged: (B,) - binary labels
        Returns:
            total_loss, loss_dict
        """
        # Core losses
        uniformity = self.uniformity_loss(embeddings, masks, is_forged)
        separation = self.separation_loss(embeddings, masks)
        
        # Optional: Keep triplet for hard mining (can help with fine details)
        triplet = self.triplet_loss(embeddings, masks) if self.config.USE_TRIPLET else 0.0
        
        # PRIORITY: Separation is 5x more important than uniformity
        total = (
            self.config.LOSS_WEIGHT_UNIFORMITY * uniformity + 
            self.config.LOSS_WEIGHT_SEPARATION * separation +
            self.config.LOSS_WEIGHT_TRIPLET * triplet
        )
        
        loss_dict = {
            'uniformity': uniformity.item() if torch.is_tensor(uniformity) else uniformity,
            'separation': separation.item() if torch.is_tensor(separation) else separation,
            'triplet': triplet.item() if torch.is_tensor(triplet) else triplet
        }
        
        return total, loss_dict

Writing losses.py


In [7]:
%%writefile train.py
"""Training Script for DINOv2 + MLP"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import numpy as np
import random
from pathlib import Path
import cv2

from config import Config
from model import create_model
from losses import CombinedLoss
from dataset import create_dataloaders
from clustering import cluster_embeddings, generate_masks_from_clusters
from utils import oF1_score

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

def save_checkpoint(model, optimizer, scheduler, scaler, epoch, best_of1, 
                   checkpoint_dir, is_best=False):
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.module.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_of1': best_of1
    }
    torch.save(checkpoint, os.path.join(checkpoint_dir, 'current_checkpoint.pth'))
    if is_best:
        torch.save(checkpoint, os.path.join(checkpoint_dir, 'best_checkpoint.pth'))

def compute_of1_score(model, dataloader, config, device, rank):
    """Compute OF1 with proper batched processing to avoid OOM"""
    model.eval()
    
    authentic_scores, forged_scores = [], []
    
    # Calculate how many batches to sample
    total_batches = len(dataloader)
    sample_batches = max(1, int(total_batches * config.VAL_SAMPLE_RATIO))
    
    if rank == 0:
        print(f"📊 Validating on {sample_batches}/{total_batches} batches")
    
    # Randomly sample batch indices
    import random
    random.seed(42)  # For reproducibility
    sampled_indices = sorted(random.sample(range(total_batches), sample_batches))
    sampled_set = set(sampled_indices)
    
    with torch.no_grad():
        batch_idx = 0
        pbar = tqdm(dataloader, desc="Validation", disable=(rank != 0), total=len(dataloader))
        
        for batch in pbar:
            # Skip batches not in our sample
            if batch_idx not in sampled_set:
                batch_idx += 1
                pbar.update(1)
                continue
            
            inputs = batch['input'].to(device)  # (B, 3, 462, 462)
            masks_33 = batch['masks'].cpu().numpy()  # (B, 6, 33, 33)
            is_forged_batch = batch['is_forged'].cpu().numpy()
            
            # Get embeddings for entire batch
            embeddings = model(inputs)  # (B, 33, 33, 512)
            embeddings_np = embeddings.cpu().numpy()
            
            # Process each sample in batch
            for b in range(inputs.shape[0]):
                # Filter based on authentic/forged ratio
                is_forged = is_forged_batch[b]
                
                # Apply ratio filtering
                if is_forged < 0.5:  # Authentic
                    if random.random() > config.VAL_AUTHENTIC_RATIO / (config.VAL_AUTHENTIC_RATIO + config.VAL_FORGED_RATIO):
                        continue
                else:  # Forged
                    if random.random() > config.VAL_FORGED_RATIO / (config.VAL_AUTHENTIC_RATIO + config.VAL_FORGED_RATIO):
                        continue
                
                embedding = embeddings_np[b]  # (33, 33, 512)
                
                # Get ground truth masks
                gt_masks = []
                for i in range(6):
                    if masks_33[b, i].sum() > 0:
                        gt_mask = cv2.resize(masks_33[b, i],
                                           (config.EVAL_MASK_SIZE, config.EVAL_MASK_SIZE),
                                           interpolation=cv2.INTER_NEAREST)
                        gt_masks.append(gt_mask)
                
                # Cluster embeddings
                cluster_labels = cluster_embeddings(embedding, config)
                
                # Generate masks WITH VALIDATION
                pred_masks = generate_masks_from_clusters(cluster_labels, config,
                                                         max_instances=config.MAX_INSTANCES,
                                                         embeddings=embedding)
                
                # Compute OF1
                if not gt_masks:
                    score = 1.0 if not pred_masks else 0.0
                else:
                    score = 0.0 if not pred_masks else oF1_score(pred_masks, gt_masks)
                
                if np.isnan(score):
                    score = 0.0
                
                if is_forged >= 0.5:
                    forged_scores.append(score)
                else:
                    authentic_scores.append(score)
            
            # Clear memory after each batch
            del inputs, embeddings, embeddings_np
            torch.cuda.empty_cache()
            
            batch_idx += 1
            pbar.update(1)
    
    model.train()
    
    if rank == 0:
        print(f"📊 Evaluated {len(authentic_scores)} authentic, {len(forged_scores)} forged")
    
    overall = np.mean(authentic_scores + forged_scores) if (authentic_scores + forged_scores) else 0.0
    auth_of1 = np.mean(authentic_scores) if authentic_scores else 0.0
    forg_of1 = np.mean(forged_scores) if forged_scores else 0.0
    
    return overall, auth_of1, forg_of1

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, 
                   epoch, config, device, rank):
    model.train()
    total_loss = 0.0
    loss_sum = {'uniformity': 0.0, 'separation': 0.0, 'triplet': 0.0}
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}", disable=(rank != 0))
    for i, batch in enumerate(pbar):
        inputs = batch['input'].to(device)
        masks = batch['masks'].to(device)
        is_forged = batch['is_forged'].to(device)  # NEW: Pass to loss
        
        with autocast(enabled=config.USE_AMP):
            embeddings = model(inputs)  # (B, 33, 33, 512)
            loss, loss_dict = criterion(embeddings, masks, is_forged)  # NEW: Added is_forged
            loss = loss / config.GRADIENT_ACCUMULATION_STEPS
        
        scaler.scale(loss).backward()
        
        if (i + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS
        for k in loss_dict:
            loss_sum[k] += loss_dict[k]
        
        if rank == 0:
            pbar.set_postfix({
                'loss': f"{loss.item()*config.GRADIENT_ACCUMULATION_STEPS:.4f}",
                **{k: f"{v:.3f}" for k, v in loss_dict.items()}
            })
    
    return total_loss / len(loader), {k: v / len(loader) for k, v in loss_sum.items()}

def train_ddp(rank, world_size, config, smoke_test=False):
    setup_ddp(rank, world_size)
    
    if smoke_test:
        config.set_smoke_test(True)
    
    if rank == 0:
        config.print_config()
    
    # Create dataloaders
    train_loader, val_loader, train_sampler, _ = create_dataloaders(config, rank, world_size)
    
    # Create model
    model = create_model(config).to(rank)
    model = DDP(model, device_ids=[rank], find_unused_parameters=False)
    
    # Get separate parameter groups for different learning rates
    mlp_params, dino_params = model.module.get_learnable_params()
    
    optimizer = AdamW([
        {'params': mlp_params, 'lr': config.MLP_LEARNING_RATE},
        {'params': dino_params, 'lr': config.DINO_LEARNING_RATE}
    ], weight_decay=config.WEIGHT_DECAY)
    
    criterion = CombinedLoss(config).to(rank)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.T_MAX, eta_min=config.ETA_MIN)
    scaler = GradScaler(enabled=config.USE_AMP)
    
    best_of1 = -1.0
    patience_counter = 0
    early_stop = False
    
    for epoch in range(1, config.NUM_EPOCHS + 1):
        if early_stop:
            if rank == 0:
                print(f"\n🛑 Early stopping triggered at epoch {epoch}")
            break
        
        train_sampler.set_epoch(epoch)
        
        # Unfreeze DINOv2 after specified epochs
        if epoch == config.ENCODER_FREEZE_EPOCHS + 1:
            model.module.unfreeze_dino()
        
        # Train
        loss, loss_dict = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, 
            scaler, epoch, config, rank, rank
        )
        
        scheduler.step()
        
        # Validate every N epochs
        if epoch % config.VAL_FREQUENCY == 0 or epoch == 1 or epoch == config.NUM_EPOCHS:
            overall, auth, forg = compute_of1_score(model, val_loader, config, rank, rank)
            
            if rank == 0:
                print(f"\n{'='*60}")
                print(f"Epoch {epoch}/{config.NUM_EPOCHS}")
                print(f"Loss: {loss:.4f} (Uniformity: {loss_dict['uniformity']:.4f}, "
                      f"Separation: {loss_dict['separation']:.4f}, Triplet: {loss_dict['triplet']:.4f})")
                print(f"OF1: Overall={overall:.4f} | Auth={auth:.4f} | Forged={forg:.4f}")
                
                # Early stopping check
                if overall > best_of1 + config.EARLY_STOPPING_MIN_DELTA:
                    best_of1 = overall
                    patience_counter = 0
                    save_checkpoint(model, optimizer, scheduler, scaler, epoch, 
                                  best_of1, config.CHECKPOINT_DIR, is_best=True)
                    print(f"✅ New best OF1: {best_of1:.4f} (patience reset)")
                else:
                    patience_counter += 1
                    print(f"⏳ No improvement. Patience: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}")
                    
                    if patience_counter >= config.EARLY_STOPPING_PATIENCE:
                        print(f"🛑 Early stopping: No improvement for {config.EARLY_STOPPING_PATIENCE} validations")
                        early_stop = True
                
                print(f"{'='*60}\n")
        
        # Save current checkpoint periodically
        if rank == 0 and epoch % 5 == 0:
            save_checkpoint(model, optimizer, scheduler, scaler, epoch, 
                          best_of1, config.CHECKPOINT_DIR, is_best=False)
        
        # Synchronize early stopping flag across all processes
        if rank == 0:
            early_stop_tensor = torch.tensor([1.0 if early_stop else 0.0], device=rank)
        else:
            early_stop_tensor = torch.tensor([0.0], device=rank)
        
        dist.broadcast(early_stop_tensor, src=0)
        early_stop = early_stop_tensor.item() > 0.5
    
    if rank == 0:
        print(f"\n🏁 Training completed! Best OF1: {best_of1:.4f}")
    
    cleanup_ddp()

def main():
    import sys
    import torch.multiprocessing as mp
    
    smoke_test = '--smoke-test' in sys.argv
    mp.spawn(train_ddp, args=(Config.WORLD_SIZE, Config, smoke_test), 
             nprocs=Config.WORLD_SIZE, join=True)

if __name__ == '__main__':
    main()

Writing train.py


In [8]:
%%writefile test.py
"""Final Testing with Best/Moderate/Worst Predictions"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import cv2
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import json

from config import Config
from model import create_model
from dataset import ForgeryDataset, get_val_transform
from clustering import cluster_embeddings, generate_masks_from_clusters
from utils import oF1_score

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12356'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

def load_hdbscan_params(output_dir):
    """Load best HDBSCAN parameters including geometry threshold"""
    path = os.path.join(output_dir, 'best_hdbscan_params.json')
    if os.path.exists(path):
        with open(path, 'r') as f:
            data = json.load(f)
        return data['min_cluster_size'], data['min_samples'], data.get('geometry_threshold', Config.GEOMETRY_THRESHOLD)
    else:
        return Config.HDBSCAN_MIN_CLUSTER_SIZE, Config.HDBSCAN_MIN_SAMPLES, Config.GEOMETRY_THRESHOLD

def load_tuning_indices(output_dir):
    """No longer needed - using batch-based sampling"""
    return []

def load_best_model(config, device):
    best_path = os.path.join(config.CHECKPOINT_DIR, 'best_checkpoint.pth')
    curr_path = os.path.join(config.CHECKPOINT_DIR, 'current_checkpoint.pth')
    path = best_path if os.path.exists(best_path) else curr_path
    
    if not os.path.exists(path):
        if device == 0:
            print("⚠️ No checkpoint found, initializing random model")
        return create_model(config).to(device)
    
    if device == 0:
        print(f"✅ Loading checkpoint: {path}")
    model = create_model(config).to(device)
    ckpt = torch.load(path, map_location=f'cuda:{device}', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def evaluate_ddp(rank, world_size, dataset, config, dataset_name, 
                save_predictions=True):
    """Evaluate with batched processing to avoid OOM"""
    sampler = DistributedSampler(dataset, num_replicas=world_size, 
                                rank=rank, shuffle=False)
    loader = DataLoader(dataset, batch_size=config.BATCH_SIZE, 
                       sampler=sampler, num_workers=4)
    
    model = load_best_model(config, rank)
    model = DDP(model, device_ids=[rank])
    model.eval()
    
    local_metrics = []
    local_predictions = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc=f"Eval {dataset_name}", disable=(rank != 0))
        for batch in pbar:
            inputs = batch['input'].to(rank)
            masks_33 = batch['masks'].cpu().numpy()
            is_forged_batch = batch['is_forged'].cpu().numpy()
            image_names = batch['image_name']
            
            # Get embeddings for batch
            embeddings = model(inputs)  # (B, 33, 33, 512)
            embeddings_np = embeddings.cpu().numpy()
            
            B = inputs.shape[0]
            for b in range(B):
                # Get ground truth masks
                gt_masks = []
                for i in range(6):
                    if masks_33[b, i].sum() > 0:
                        gt_mask = cv2.resize(masks_33[b, i],
                                           (config.EVAL_MASK_SIZE, config.EVAL_MASK_SIZE),
                                           interpolation=cv2.INTER_NEAREST)
                        gt_masks.append(gt_mask)
                
                # Cluster and predict WITH VALIDATION
                cluster_labels = cluster_embeddings(embeddings_np[b], config)
                pred_masks = generate_masks_from_clusters(cluster_labels, config,
                                                         max_instances=config.MAX_INSTANCES,
                                                         embeddings=embeddings_np[b])
                
                # Compute OF1
                if not gt_masks:
                    score = 1.0 if not pred_masks else 0.0
                else:
                    score = 0.0 if not pred_masks else oF1_score(pred_masks, gt_masks)
                
                if np.isnan(score):
                    score = 0.0
                
                is_forged = int(is_forged_batch[b])
                
                local_metrics.append({
                    'is_forged': is_forged,
                    'pred_forged': int(len(pred_masks) > 0),
                    'of1': float(score),
                    'image_name': image_names[b]
                })
                
                if save_predictions and is_forged:
                    img_thumb = inputs[b].cpu().permute(1, 2, 0).numpy()
                    img_thumb = (img_thumb * 255).astype(np.uint8)
                    img_thumb = cv2.resize(img_thumb, (224, 224))
                    
                    local_predictions.append({
                        'of1': float(score),
                        'preds': pred_masks,
                        'gts': gt_masks,
                        'img_thumb': img_thumb,
                        'image_name': image_names[b]
                    })
            
            # Clear memory
            del inputs, embeddings, embeddings_np
            torch.cuda.empty_cache()
    
    # Gather metrics
    gathered_metrics = [None] * world_size
    dist.all_gather_object(gathered_metrics, local_metrics)
    
    # Gather predictions
    gathered_preds = [None] * world_size
    dist.all_gather_object(gathered_preds, local_predictions)
    
    final_metrics = None
    final_predictions = None
    
    if rank == 0:
        all_data = [x for sub in gathered_metrics for x in sub]
        
        auth = [r for r in all_data if r['is_forged'] == 0]
        forg = [r for r in all_data if r['is_forged'] == 1]
        
        of1_all = np.mean([r['of1'] for r in all_data]) if all_data else 0.0
        of1_auth = np.mean([r['of1'] for r in auth]) if auth else 0.0
        of1_forg = np.mean([r['of1'] for r in forg]) if forg else 0.0
        
        y_true = [r['is_forged'] for r in all_data]
        y_pred = [r['pred_forged'] for r in all_data]
        
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        
        final_metrics = {
            'segmentation': {
                'overall_of1': of1_all,
                'authentic_of1': of1_auth,
                'forged_of1': of1_forg
            },
            'detection': {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision': precision_score(y_true, y_pred, zero_division=0),
                'recall': recall_score(y_true, y_pred, zero_division=0),
                'f1': f1_score(y_true, y_pred, zero_division=0),
                'auc': roc_auc_score(y_true, y_pred) if len(set(y_true)) > 1 else 0.0,
                'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 
                                    'fn': int(fn), 'tp': int(tp)}
            }
        }
        
        # Save confusion matrix
        plt.figure(figsize=(5, 4))
        sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Auth', 'Forg'], yticklabels=['Auth', 'Forg'])
        plt.title(f'{dataset_name} Confusion Matrix')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.savefig(os.path.join(config.OUTPUT_DIR, 
                                f'cm_{dataset_name.replace(" ", "_")}.png'))
        plt.close()
        
        if save_predictions:
            all_preds = [x for sub in gathered_preds for x in sub]
            final_predictions = all_preds
    
    return final_metrics, final_predictions

def visualize_predictions(predictions, output_dir, config):
    """Save best, moderate, and worst predictions"""
    if not predictions:
        return
    
    # Sort by OF1 score
    sorted_preds = sorted(predictions, key=lambda x: x['of1'])
    
    n_save = config.SAVE_TOP_PREDICTIONS
    
    # Get best, moderate, worst
    worst = sorted_preds[:n_save]
    best = sorted_preds[-n_save:]
    
    # Moderate: middle samples
    mid_start = (len(sorted_preds) - n_save) // 2
    moderate = sorted_preds[mid_start:mid_start + n_save] if len(sorted_preds) >= 3 * n_save else []
    
    categories = [('worst', worst), ('moderate', moderate), ('best', best)]
    
    for cat_name, pred_list in categories:
        if not pred_list:
            continue
        
        vis_dir = os.path.join(output_dir, f'{cat_name}_predictions')
        Path(vis_dir).mkdir(parents=True, exist_ok=True)
        
        print(f"🖼️ Saving {len(pred_list)} {cat_name} predictions...")
        
        for i, res in enumerate(pred_list):
            fig, ax = plt.subplots(1, 3, figsize=(12, 4))
            
            # Original image
            ax[0].imshow(res['img_thumb'])
            ax[0].set_title(f"{res['image_name'][:20]}...")
            ax[0].axis('off')
            
            # Ground truth overlay
            gtv = res['img_thumb'].copy()
            for j, m in enumerate(res['gts']):
                color_mask = np.zeros_like(gtv)
                color_mask[:, :, j % 3] = (m * 255).astype(np.uint8)
                gtv = cv2.addWeighted(gtv, 1.0, color_mask, 0.4, 0)
            ax[1].imshow(gtv)
            ax[1].set_title(f'GT ({len(res["gts"])} instances)')
            ax[1].axis('off')
            
            # Prediction overlay
            prv = res['img_thumb'].copy()
            for j, m in enumerate(res['preds']):
                color_mask = np.zeros_like(prv)
                color_mask[:, :, j % 3] = (m * 255).astype(np.uint8)
                prv = cv2.addWeighted(prv, 1.0, color_mask, 0.4, 0)
            ax[2].imshow(prv)
            ax[2].set_title(f'Pred (OF1: {res["of1"]:.3f})')
            ax[2].axis('off')
            
            plt.tight_layout()
            plt.savefig(os.path.join(vis_dir, f'{cat_name}_{i:03d}.png'))
            plt.close()

def run_test(rank, world_size):
    setup_ddp(rank, world_size)
    
    if rank == 0:
        Path(Config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    dist.barrier()
    
    # Load best HDBSCAN parameters including geometry threshold
    if rank == 0:
        min_cs, min_s, geo_t = load_hdbscan_params(Config.OUTPUT_DIR)
        print(f"📊 Using tuned parameters:")
        print(f"   min_cluster_size = {min_cs}")
        print(f"   min_samples = {min_s}")
        print(f"   geometry_threshold = {geo_t:.2f}")
        Config.HDBSCAN_MIN_CLUSTER_SIZE = min_cs
        Config.HDBSCAN_MIN_SAMPLES = min_s
        Config.GEOMETRY_THRESHOLD = geo_t
    dist.barrier()
    
    # Create test datasets (full datasets, no filtering)
    test_ds = ForgeryDataset(Config.TEST_AUTHENTIC_DIR, Config.TEST_FORGED_DIR,
                            Config.TEST_MASK_DIR, get_val_transform(Config), Config)
    
    hq_ds = ForgeryDataset(Config.HQ_AUTHENTIC_DIR, Config.HQ_FORGED_DIR,
                          Config.HQ_MASK_DIR, get_val_transform(Config), Config)
    
    if rank == 0:
        print(f"📊 Evaluating on full test set ({len(test_ds)} samples)")
    
    # Evaluate
    test_metrics, test_preds = evaluate_ddp(rank, world_size, test_ds, 
                                           Config, "Test Set", True)
    hq_metrics, _ = evaluate_ddp(rank, world_size, hq_ds, Config, "HQ Set", False)
    
    if rank == 0:
        print("\n" + "=" * 60)
        print("FINAL EVALUATION REPORT")
        print("=" * 60)
        
        for name, metrics in [("TEST SET", test_metrics), ("HQ SET", hq_metrics)]:
            s = metrics['segmentation']
            d = metrics['detection']
            c = d['confusion_matrix']
            
            print(f"\n🔹 {name}")
            print(f"   Segmentation OF1:")
            print(f"      Overall:    {s['overall_of1']:.4f}")
            print(f"      Authentic:  {s['authentic_of1']:.4f}")
            print(f"      Forged:     {s['forged_of1']:.4f}")
            print(f"   Detection:")
            print(f"      Accuracy:   {d['accuracy']:.4f}")
            print(f"      Precision:  {d['precision']:.4f}")
            print(f"      Recall:     {d['recall']:.4f}")
            print(f"      F1:         {d['f1']:.4f}")
            print(f"      AUC:        {d['auc']:.4f}")
            print(f"   Confusion Matrix:")
            print(f"      TN: {c['tn']:4d}  FP: {c['fp']:4d}")
            print(f"      FN: {c['fn']:4d}  TP: {c['tp']:4d}")
        
        # Save metrics
        with open(os.path.join(Config.OUTPUT_DIR, 'final_metrics.json'), 'w') as f:
            json.dump({'test': test_metrics, 'hq': hq_metrics}, f, indent=2)
        
        # Visualize predictions
        if test_preds:
            visualize_predictions(test_preds, Config.OUTPUT_DIR, Config)
        
        print(f"\n✅ Evaluation complete! Results saved to {Config.OUTPUT_DIR}")
    
    cleanup_ddp()

if __name__ == '__main__':
    import torch.multiprocessing as mp
    mp.spawn(run_test, args=(Config.WORLD_SIZE,), 
             nprocs=Config.WORLD_SIZE, join=True)

Writing test.py


In [9]:
%%writefile tuning.py
"""HDBSCAN + Geometry Threshold Parameter Tuning - OPTIMIZED VERSION"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, ConcatDataset
from torch.utils.data.distributed import DistributedSampler
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import random
from pathlib import Path
import cv2
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

# Suppress sklearn warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')
warnings.filterwarnings('ignore', message='.*force_all_finite.*')

from config import Config
from model import create_model
from dataset import ForgeryDataset, get_val_transform
from finetune_dataset import RepeatedDataset, SubsampledDataset, get_standard_augmentation
from clustering import cluster_embeddings
from utils import oF1_score

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12357'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

def load_model(config, device):
    path = os.path.join(config.CHECKPOINT_DIR, 'best_checkpoint.pth')
    if not os.path.exists(path):
        path = os.path.join(config.CHECKPOINT_DIR, 'current_checkpoint.pth')
    
    ckpt = torch.load(path, map_location=f'cuda:{device}', weights_only=False)
    model = create_model(config).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def upsample_mask_patchwise(mask_33, patch_size=14, target_size=462):
    """Upsample 33x33 mask to target_size"""
    H, W = mask_33.shape
    mask_target = np.zeros((target_size, target_size), dtype=np.float32)
    
    for i in range(H):
        for j in range(W):
            if mask_33[i, j] > 0.5:
                y_start = i * patch_size
                y_end = min((i + 1) * patch_size, target_size)
                x_start = j * patch_size
                x_end = min((j + 1) * patch_size, target_size)
                mask_target[y_start:y_end, x_start:x_end] = 1.0
    
    return mask_target

def predict_with_params_single(args):
    """
    Wrapper for parallel processing - predicts a single sample
    Returns: score
    """
    embeddings, gt_masks, min_cluster_size, min_samples, geometry_threshold, config_dict = args
    
    # Suppress warnings in worker process
    warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')
    
    from hdbscan import HDBSCAN
    
    H, W, D = embeddings.shape
    emb_flat = embeddings.reshape(-1, D)
    
    # Run HDBSCAN
    clusterer = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=config_dict['metric'],
        cluster_selection_method=config_dict['selection_method'],
        core_dist_n_jobs=-1
    )
    
    cluster_labels = clusterer.fit_predict(emb_flat)
    cluster_labels = cluster_labels.reshape(H, W)
    
    # Get unique clusters
    unique_clusters = np.unique(cluster_labels)
    unique_clusters = unique_clusters[unique_clusters >= 0]
    
    if len(unique_clusters) == 0:
        return 1.0 if not gt_masks else 0.0
    
    # Count cluster sizes to identify background
    label_flat = cluster_labels.flatten()
    cluster_sizes = [(c, (label_flat == c).sum()) for c in unique_clusters]
    cluster_sizes.sort(key=lambda x: x[1], reverse=True)
    
    background_cluster = cluster_sizes[0][0]
    candidate_clusters = [c for c, _ in cluster_sizes[1:]]
    
    # VALIDATE CANDIDATES
    if len(candidate_clusters) > 0:
        bg_mask = (label_flat == background_cluster)
        bg_embeddings = emb_flat[bg_mask]
        bg_centroid = bg_embeddings.mean(axis=0)
        
        valid_clusters = []
        for cluster_id in candidate_clusters:
            cand_mask = (label_flat == cluster_id)
            cand_embeddings = emb_flat[cand_mask]
            
            if cand_embeddings.shape[0] == 0:
                continue
            
            cand_centroid = cand_embeddings.mean(axis=0)
            
            # Compute cosine distance
            bg_norm = bg_centroid / (np.linalg.norm(bg_centroid) + 1e-8)
            cand_norm = cand_centroid / (np.linalg.norm(cand_centroid) + 1e-8)
            cosine_sim = np.dot(bg_norm, cand_norm)
            cosine_dist = 1.0 - cosine_sim
            
            if cosine_dist > geometry_threshold:
                valid_clusters.append(cluster_id)
        
        forgery_clusters = valid_clusters
    else:
        forgery_clusters = []
    
    # Limit to max instances
    if len(forgery_clusters) > config_dict['max_instances']:
        valid_sizes = [(c, (label_flat == c).sum()) for c in forgery_clusters]
        valid_sizes.sort(key=lambda x: x[1], reverse=True)
        forgery_clusters = [c for c, _ in valid_sizes[:config_dict['max_instances']]]
    
    # Generate masks
    if len(forgery_clusters) == 0:
        pred_masks = []
    else:
        pred_masks = []
        for cluster_id in forgery_clusters:
            mask = (cluster_labels == cluster_id).astype(np.float32)
            mask_462 = upsample_mask_patchwise(mask, patch_size=14, target_size=462)
            mask_224 = cv2.resize(mask_462, (config_dict['eval_mask_size'], 
                                            config_dict['eval_mask_size']), 
                                 interpolation=cv2.INTER_NEAREST)
            pred_masks.append(mask_224)
    
    # Compute OF1
    if not gt_masks:
        score = 1.0 if not pred_masks else 0.0
    else:
        score = 0.0 if not pred_masks else oF1_score(pred_masks, gt_masks)
    
    return score if not np.isnan(score) else 0.0

def predict_with_params_parallel(all_data, min_cluster_size, min_samples, 
                                 geometry_threshold, config, max_workers=None):
    """
    Predict with parameters using multiprocessing for parallel HDBSCAN
    """
    if max_workers is None:
        max_workers = min(os.cpu_count() or 1, len(all_data))
    
    # Create config dict for serialization
    config_dict = {
        'metric': config.HDBSCAN_METRIC,
        'selection_method': config.HDBSCAN_CLUSTER_SELECTION_METHOD,
        'max_instances': config.MAX_INSTANCES,
        'eval_mask_size': config.EVAL_MASK_SIZE
    }
    
    # Prepare arguments
    args_list = [
        (data['embeddings'], data['gt_masks'], min_cluster_size, 
         min_samples, geometry_threshold, config_dict)
        for data in all_data
    ]
    
    # Run in parallel with progress bar
    scores = []
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(predict_with_params_single, args) 
                  for args in args_list]
        
        for future in tqdm(as_completed(futures), total=len(futures), 
                          desc=f"  Testing ({min_cluster_size},{min_samples},{geometry_threshold:.2f})",
                          leave=False, disable=True):  # Disable inner progress
            scores.append(future.result())
    
    return np.mean(scores) if scores else 0.0

def smart_grid_search(all_data, config, rank):
    """
    Smart grid search using gradient-like approach to find optimal parameters
    
    Strategy:
    1. Start with coarse grid
    2. Find best region
    3. Refine search in that region (like zooming in)
    4. Use local optimization
    """
    if rank == 0:
        print(f"\n{'='*60}")
        print("SMART ADAPTIVE GRID SEARCH")
        print(f"{'='*60}")
    
    # ========== PHASE 1: Coarse Grid Search ==========
    if rank == 0:
        print(f"\n🔍 PHASE 1: Coarse Grid Search")
    
    # Coarse grid
    coarse_cs = Config.TEST_MIN_CLUSTER_SIZES
    coarse_s = Config.TEST_MIN_SAMPLES
    coarse_geo = [0.3, 0.5, 0.7]
    
    best_score = -1
    best_params = None
    all_scores = {}
    
    total_combos = len(coarse_cs) * len(coarse_s) * len(coarse_geo)
    
    with tqdm(total=total_combos, desc="Coarse Search", disable=(rank != 0)) as pbar:
        for min_cs in coarse_cs:
            for min_s in coarse_s:
                for geo_t in coarse_geo:
                    score = predict_with_params_parallel(
                        all_data, min_cs, min_s, geo_t, config, max_workers=4
                    )
                    
                    all_scores[(min_cs, min_s, geo_t)] = score
                    
                    if score > best_score:
                        best_score = score
                        best_params = (min_cs, min_s, geo_t)
                        if rank == 0:
                            pbar.set_postfix({'best': f'{best_score:.4f}', 
                                            'params': f'({min_cs},{min_s},{geo_t:.1f})'})
                    
                    pbar.update(1)
    
    if rank == 0:
        print(f"   ✅ Coarse Best: {best_params} → {best_score:.4f}")
    
    # ========== PHASE 2: Fine Grid Search Around Best ==========
    if rank == 0:
        print(f"\n🔍 PHASE 2: Fine Grid Search (Zoom In)")
    
    best_cs, best_s, best_geo = best_params
    
    # Fine grid around best
    fine_cs = [max(10, best_cs - 5), best_cs, min(30, best_cs + 5)]
    fine_s = [max(3, best_s - 2), best_s, min(15, best_s + 2)]
    fine_geo = [max(0.2, best_geo - 0.1), best_geo, min(0.8, best_geo + 0.1)]
    
    # Remove duplicates
    fine_cs = sorted(list(set(fine_cs)))
    fine_s = sorted(list(set(fine_s)))
    fine_geo = sorted(list(set(fine_geo)))
    
    fine_combos = len(fine_cs) * len(fine_s) * len(fine_geo)
    
    with tqdm(total=fine_combos, desc="Fine Search", disable=(rank != 0)) as pbar:
        for min_cs in fine_cs:
            for min_s in fine_s:
                for geo_t in fine_geo:
                    if (min_cs, min_s, geo_t) in all_scores:
                        pbar.update(1)
                        continue
                    
                    score = predict_with_params_parallel(
                        all_data, min_cs, min_s, geo_t, config, max_workers=4
                    )
                    
                    all_scores[(min_cs, min_s, geo_t)] = score
                    
                    if score > best_score:
                        best_score = score
                        best_params = (min_cs, min_s, geo_t)
                        if rank == 0:
                            pbar.set_postfix({'best': f'{best_score:.4f}',
                                            'params': f'({min_cs},{min_s},{geo_t:.2f})'})
                    
                    pbar.update(1)
    
    if rank == 0:
        print(f"   ✅ Fine Best: {best_params} → {best_score:.4f}")
    
    # ========== PHASE 3: Local Gradient-Like Refinement ==========
    if rank == 0:
        print(f"\n🔍 PHASE 3: Local Refinement (Gradient Descent)")
    
    best_cs, best_s, best_geo = best_params
    improved = True
    iteration = 0
    max_iterations = 10
    
    pbar = tqdm(total=max_iterations, desc="Local Refinement", disable=(rank != 0))
    
    while improved and iteration < max_iterations:
        improved = False
        iteration += 1
        
        # Test all 6 neighbors (±1 step in each dimension)
        neighbors = [
            (max(10, best_cs - 5), best_s, best_geo),
            (min(30, best_cs + 5), best_s, best_geo),
            (best_cs, max(3, best_s - 2), best_geo),
            (best_cs, min(15, best_s + 2), best_geo),
            (best_cs, best_s, max(0.2, best_geo - 0.05)),
            (best_cs, best_s, min(0.8, best_geo + 0.05)),
        ]
        
        for min_cs, min_s, geo_t in neighbors:
            if (min_cs, min_s, geo_t) in all_scores:
                score = all_scores[(min_cs, min_s, geo_t)]
            else:
                score = predict_with_params_parallel(
                    all_data, min_cs, min_s, geo_t, config, max_workers=4
                )
                all_scores[(min_cs, min_s, geo_t)] = score
            
            if score > best_score:
                best_score = score
                best_params = (min_cs, min_s, geo_t)
                improved = True
                if rank == 0:
                    pbar.set_postfix({'best': f'{best_score:.4f}',
                                    'params': f'{best_params}'})
        
        pbar.update(1)
        
        if improved:
            best_cs, best_s, best_geo = best_params
    
    pbar.close()
    
    if rank == 0:
        print(f"   ✅ Final Best: {best_params} → {best_score:.4f}")
        print(f"   Converged in {iteration} iterations")
    
    return best_params[0], best_params[1], best_params[2], best_score, all_scores

def create_tuning_dataset(config, rank):
    """
    Create tuning dataset: 400% HQ + 10% Test
    Distribution: 30% authentic, 70% forged
    """
    if rank == 0:
        print(f"\n📦 Creating Tuning Dataset...")
    
    # 1. HQ Data (48 images × 4 = 192 samples) - ONLY FORGED
    hq_base_ds = ForgeryDataset(
        config.HQ_AUTHENTIC_DIR,  # Empty
        config.HQ_FORGED_DIR,
        config.HQ_MASK_DIR,
        get_standard_augmentation(config),
        config
    )
    hq_repeated_ds = RepeatedDataset(hq_base_ds, 4, seed=999)
    
    # 2. Test Data (10% = ~500 samples)
    test_base_ds = ForgeryDataset(
        config.TEST_AUTHENTIC_DIR,
        config.TEST_FORGED_DIR,
        config.TEST_MASK_DIR,
        None,  # No augmentation
        config
    )
    test_subsample_ds = SubsampledDataset(test_base_ds, 0.10, seed=999)
    
    # Combine datasets
    combined_ds = ConcatDataset([hq_repeated_ds, test_subsample_ds])
    
    if rank == 0:
        print(f"   HQ:   {len(hq_repeated_ds)} samples (48 × 4)")
        print(f"   Test: {len(test_subsample_ds)} samples (10%)")
        print(f"   TOTAL: {len(combined_ds)} samples")
    
    return combined_ds

def threshold_search_ddp(rank, world_size):
    setup_ddp(rank, world_size)
    
    # Create tuning dataset
    ds = create_tuning_dataset(Config, rank)
    
    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=False)
    loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=4)
    
    model = load_model(Config, rank)
    model = DDP(model, device_ids=[rank])
    
    if rank == 0:
        print(f"\n📊 Total batches: {len(loader)}")
    
    # ========== EXTRACT EMBEDDINGS (DO ONCE) ==========
    all_data = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting embeddings", disable=(rank != 0)):
            inputs = batch['input'].to(rank)
            masks_33 = batch['masks'].cpu().numpy()
            is_forged_batch = batch['is_forged'].cpu().numpy()
            
            embeddings = model(inputs)
            embeddings_np = embeddings.cpu().numpy()
            
            for b in range(inputs.shape[0]):
                is_forged = is_forged_batch[b]
                
                # Apply 30/70 distribution filter
                if is_forged < 0.5:  # Authentic
                    if random.random() > Config.TUNING_AUTHENTIC_RATIO:
                        continue
                else:  # Forged
                    if random.random() > Config.TUNING_FORGED_RATIO:
                        continue
                
                gt_masks = []
                for i in range(6):
                    if masks_33[b, i].sum() > 0:
                        gt_mask = cv2.resize(masks_33[b, i],
                                           (Config.EVAL_MASK_SIZE, Config.EVAL_MASK_SIZE),
                                           interpolation=cv2.INTER_NEAREST)
                        gt_masks.append(gt_mask)
                
                all_data.append({
                    'embeddings': embeddings_np[b],
                    'gt_masks': gt_masks
                })
            
            del inputs, embeddings, embeddings_np
            torch.cuda.empty_cache()
    
    if rank == 0:
        print(f"📊 Collected {len(all_data)} samples for tuning")
    
    # ========== SMART PARAMETER SEARCH ==========
    best_min_cs, best_min_s, best_geo_val, best_score, all_scores = \
        smart_grid_search(all_data, Config, rank)
    
    # Gather results from all processes
    gathered = [None for _ in range(world_size)]
    local_results = {
        'min_cs': best_min_cs,
        'min_s': best_min_s,
        'geo_t': best_geo_val,
        'score': best_score
    }
    dist.all_gather_object(gathered, local_results)
    
    if rank == 0:
        # Find overall best across all ranks
        best_overall = max(gathered, key=lambda x: x['score'])
        
        print(f"\n{'='*60}")
        print("🏆 BEST CONFIGURATION FOUND")
        print(f"{'='*60}")
        print(f"   min_cluster_size:     {best_overall['min_cs']}")
        print(f"   min_samples:          {best_overall['min_s']}")
        print(f"   geometry_threshold:   {best_overall['geo_t']:.2f}")
        print(f"   Tuning OF1 Score:     {best_overall['score']:.4f}")
        print(f"{'='*60}")
        
        # Save best parameters
        Path(Config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        best_params = {
            'min_cluster_size': int(best_overall['min_cs']),
            'min_samples': int(best_overall['min_s']),
            'geometry_threshold': float(best_overall['geo_t']),
            'tuning_of1': float(best_overall['score'])
        }
        
        with open(os.path.join(Config.OUTPUT_DIR, 'best_hdbscan_params.json'), 'w') as f:
            json.dump(best_params, f, indent=2)
        
        # Plot 3D parameter space
        if len(all_scores) > 10:
            from mpl_toolkits.mplot3d import Axes3D
            
            params = list(all_scores.keys())
            scores = list(all_scores.values())
            
            cs_vals = [p[0] for p in params]
            s_vals = [p[1] for p in params]
            geo_vals = [p[2] for p in params]
            
            fig = plt.figure(figsize=(15, 5))
            
            # CS vs S
            ax1 = fig.add_subplot(131)
            scatter1 = ax1.scatter(cs_vals, s_vals, c=scores, cmap='viridis', s=100)
            ax1.scatter([best_overall['min_cs']], [best_overall['min_s']], 
                       color='red', s=200, marker='*', label='Best')
            ax1.set_xlabel('min_cluster_size')
            ax1.set_ylabel('min_samples')
            ax1.set_title('HDBSCAN Parameters')
            ax1.legend()
            plt.colorbar(scatter1, ax=ax1, label='OF1')
            
            # CS vs Geo
            ax2 = fig.add_subplot(132)
            scatter2 = ax2.scatter(cs_vals, geo_vals, c=scores, cmap='viridis', s=100)
            ax2.scatter([best_overall['min_cs']], [best_overall['geo_t']], 
                       color='red', s=200, marker='*', label='Best')
            ax2.set_xlabel('min_cluster_size')
            ax2.set_ylabel('geometry_threshold')
            ax2.set_title('Cluster Size vs Geometry')
            ax2.legend()
            plt.colorbar(scatter2, ax=ax2, label='OF1')
            
            # S vs Geo
            ax3 = fig.add_subplot(133)
            scatter3 = ax3.scatter(s_vals, geo_vals, c=scores, cmap='viridis', s=100)
            ax3.scatter([best_overall['min_s']], [best_overall['geo_t']], 
                       color='red', s=200, marker='*', label='Best')
            ax3.set_xlabel('min_samples')
            ax3.set_ylabel('geometry_threshold')
            ax3.set_title('Samples vs Geometry')
            ax3.legend()
            plt.colorbar(scatter3, ax=ax3, label='OF1')
            
            plt.tight_layout()
            plt.savefig(os.path.join(Config.OUTPUT_DIR, 'parameter_space.png'), dpi=150)
            plt.close()
        
        print(f"\n✅ Tuning complete. Best params saved to {Config.OUTPUT_DIR}")
    
    cleanup_ddp()

if __name__ == '__main__':
    import torch.multiprocessing as mp
    mp.spawn(threshold_search_ddp, args=(Config.WORLD_SIZE,), 
             nprocs=Config.WORLD_SIZE, join=True)

Writing tuning.py


In [10]:
# !python train.py --smoke-test

In [11]:
# !python train.py

In [12]:
# !python test.py

In [13]:
# import shutil
# import os

# # Source directory
# dir_path = "/kaggle/working/checkpoints"
# # Output filename (without extension)
# output_path = "/kaggle/working/checkpoints_archive"

# # Create the zip file
# shutil.make_archive(output_path, 'zip', dir_path)

# print(f"Successfully created: {output_path}.zip")

In [14]:
%%writefile finetune_config.py
"""Fine-tuning Configuration for Final Submission"""
import torch
from config import Config as BaseConfig

class FinetuneConfig(BaseConfig):
    # ==================== FINE-TUNING SPECIFIC ====================
    NUM_EPOCHS = 30
    ENCODER_FREEZE_EPOCHS = 0  # Don't freeze, we're fine-tuning
    
    # Lower learning rates for fine-tuning
    MLP_LEARNING_RATE = 1e-6    # 10x lower
    DINO_LEARNING_RATE = 1e-6     # 10x lower
    WEIGHT_DECAY = 1e-4
    
    # Batch size (can increase if memory allows)
    BATCH_SIZE = 24
    GRADIENT_ACCUMULATION_STEPS = 2
    
    # Validation
    VAL_FREQUENCY = 3  # Every 3 epochs
    
    # ==================== DATASET COMPOSITION ====================
    # Training composition
    TEST_DATA_RATIO = 1.0          # 100% of test data (5k images)
    HQ_DATA_MULTIPLIER = 6         # 48 images × 6 = 288 samples per epoch
    TRAIN_DATA_RATIO = 0.3         # 30% of train data (~7.2k images)
    
    # Validation composition
    VAL_TEST_SAMPLE_RATIO = 0.1    # 10% of test data (500 images)
    VAL_HQ_MULTIPLIER = 4          # 48 × 4 = 192 samples
    
    # Validation distribution (must match test distribution)
    VAL_AUTHENTIC_RATIO = 0.30
    VAL_FORGED_RATIO = 0.70
    
    # ==================== AUGMENTATION ====================
    # Heavy augmentation for HQ data
    HQ_AUG_ROTATION_LIMIT = 45
    HQ_AUG_PROB_FLIP = 0.7
    HQ_AUG_PROB_ROTATE = 0.7
    HQ_AUG_BRIGHTNESS = 0.3
    HQ_AUG_CONTRAST = 0.2
    HQ_AUG_SATURATION = 0.2
    HQ_AUG_HUE = 0.08
    HQ_AUG_PROB_COLOR = 0.7
    HQ_AUG_PROB_BLUR = 0.5
    HQ_AUG_PROB_NOISE = 0.5
    
    # ==================== EARLY STOPPING ====================
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.0005
    
    # ==================== PATHS ====================
    FINETUNE_CHECKPOINT_DIR = "/kaggle/working/finetune_checkpoints"
    FINETUNE_OUTPUT_DIR = "/kaggle/working/finetune_outputs"
    
    # Load best model from initial training
    PRETRAINED_CHECKPOINT = "/kaggle/input/sci-forge-dino-sim/pytorch/default/1/best_checkpoint.pth"
    
    @classmethod
    def print_config(cls):
        print("=" * 60)
        print("FINE-TUNING CONFIGURATION")
        print("=" * 60)
        print(f"Training Data Composition:")
        print(f"  - Test Data: {cls.TEST_DATA_RATIO*100}% (5k images)")
        print(f"  - HQ Data: 48 × {cls.HQ_DATA_MULTIPLIER} = {48*cls.HQ_DATA_MULTIPLIER} samples/epoch")
        print(f"  - Train Data: {cls.TRAIN_DATA_RATIO*100}% (~7.2k images)")
        print(f"  - Total: ~{5000 + 48*cls.HQ_DATA_MULTIPLIER + 7200} samples/epoch")
        print(f"\nValidation Data Composition:")
        print(f"  - Test Data: {cls.VAL_TEST_SAMPLE_RATIO*100}% (500 images)")
        print(f"  - HQ Data: 48 × {cls.VAL_HQ_MULTIPLIER} = {48*cls.VAL_HQ_MULTIPLIER} samples")
        print(f"  - Distribution: {cls.VAL_AUTHENTIC_RATIO*100}% Auth / {cls.VAL_FORGED_RATIO*100}% Forged")
        print(f"\nLearning Rates:")
        print(f"  - MLP: {cls.MLP_LEARNING_RATE}")
        print(f"  - DINOv2: {cls.DINO_LEARNING_RATE}")
        print(f"\nTraining:")
        print(f"  - Epochs: {cls.NUM_EPOCHS}")
        print(f"  - Validate every: {cls.VAL_FREQUENCY} epochs")
        print("=" * 60)

Writing finetune_config.py


In [15]:
%%writefile finetune_dataset.py
"""Fine-tuning Dataset with Mixed Sources"""
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.utils.data.distributed import DistributedSampler
import albumentations as A
import cv2

from dataset import ForgeryDataset

class RepeatedDataset(Dataset):
    """Repeat a dataset multiple times (for HQ data augmentation)"""
    def __init__(self, base_dataset, multiplier, seed=42):
        self.base_dataset = base_dataset
        self.multiplier = multiplier
        self.seed = seed
        
    def __len__(self):
        return len(self.base_dataset) * self.multiplier
    
    def __getitem__(self, idx):
        # Different augmentation each time due to random seed
        base_idx = idx % len(self.base_dataset)
        repeat_num = idx // len(self.base_dataset)
        
        # Set different seed for each repeat
        random.seed(self.seed + idx)
        np.random.seed(self.seed + idx)
        
        return self.base_dataset[base_idx]

class SubsampledDataset(Dataset):
    """Subsample a dataset to a specific ratio"""
    def __init__(self, base_dataset, sample_ratio, seed=42):
        self.base_dataset = base_dataset
        self.sample_ratio = sample_ratio
        
        # Randomly sample indices
        random.seed(seed)
        total = len(base_dataset)
        n_samples = int(total * sample_ratio)
        self.indices = random.sample(range(total), n_samples)
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        return self.base_dataset[self.indices[idx]]

def get_heavy_augmentation(config):
    """Heavy augmentation for HQ data"""
    return A.Compose([
        A.HorizontalFlip(p=config.HQ_AUG_PROB_FLIP),
        A.VerticalFlip(p=config.HQ_AUG_PROB_FLIP),
        A.Rotate(limit=config.HQ_AUG_ROTATION_LIMIT, p=config.HQ_AUG_PROB_ROTATE),
        A.GaussianBlur(blur_limit=config.AUG_BLUR_LIMIT, p=config.HQ_AUG_PROB_BLUR),
        A.GaussNoise(var_limit=config.AUG_NOISE_VAR_LIMIT, p=config.HQ_AUG_PROB_NOISE),
        A.ImageCompression(quality_range=config.AUG_JPEG_QUALITY_RANGE, 
                          p=config.AUG_PROB_JPEG),
        A.ColorJitter(
            brightness=config.HQ_AUG_BRIGHTNESS,
            contrast=config.HQ_AUG_CONTRAST,
            saturation=config.HQ_AUG_SATURATION,
            hue=config.HQ_AUG_HUE,
            p=config.HQ_AUG_PROB_COLOR
        ),
        # Additional heavy augmentations
        A.RandomBrightnessContrast(p=0.5),
        A.RandomGamma(p=0.3),
        A.Sharpen(p=0.3),
    ])

def get_standard_augmentation(config):
    """Standard augmentation for test/train data"""
    from dataset import get_train_transform
    return get_train_transform(config)

def create_finetune_dataloaders(config, rank=0, world_size=1):
    """
    Create fine-tuning dataloaders with mixed sources
    
    Training: 100% test + 600% HQ + 30% train
    Validation: 10% test + 400% HQ (with 30/70 auth/forged distribution)
    """
    
    # ========== TRAINING DATASETS ==========
    
    # 1. Test data (100%)
    test_train_ds = ForgeryDataset(
        config.TEST_AUTHENTIC_DIR,
        config.TEST_FORGED_DIR,
        config.TEST_MASK_DIR,
        get_standard_augmentation(config),
        config
    )
    
    # 2. HQ data (48 images × 6 = 288 samples)
    hq_base_ds = ForgeryDataset(
        config.HQ_AUTHENTIC_DIR,  # Empty for HQ (only forged)
        config.HQ_FORGED_DIR,
        config.HQ_MASK_DIR,
        get_heavy_augmentation(config),
        config
    )
    hq_train_ds = RepeatedDataset(hq_base_ds, config.HQ_DATA_MULTIPLIER, seed=42)
    
    # 3. Train data (30%)
    train_base_ds = ForgeryDataset(
        config.TRAIN_AUTHENTIC_DIR,
        config.TRAIN_FORGED_DIR,
        config.TRAIN_MASK_DIR,
        get_standard_augmentation(config),
        config
    )
    train_subsample_ds = SubsampledDataset(train_base_ds, config.TRAIN_DATA_RATIO, seed=42)
    
    # Combine all training datasets
    combined_train_ds = ConcatDataset([test_train_ds, hq_train_ds, train_subsample_ds])
    
    if rank == 0:
        print(f"📦 Fine-tuning Training Data:")
        print(f"   Test:  {len(test_train_ds)} samples")
        print(f"   HQ:    {len(hq_train_ds)} samples (48 × {config.HQ_DATA_MULTIPLIER})")
        print(f"   Train: {len(train_subsample_ds)} samples ({config.TRAIN_DATA_RATIO*100}%)")
        print(f"   TOTAL: {len(combined_train_ds)} samples")
    
    # ========== VALIDATION DATASETS ==========
    
    # 1. Test data (10% subsampled)
    test_val_base_ds = ForgeryDataset(
        config.TEST_AUTHENTIC_DIR,
        config.TEST_FORGED_DIR,
        config.TEST_MASK_DIR,
        None,  # No augmentation for validation
        config
    )
    test_val_ds = SubsampledDataset(test_val_base_ds, config.VAL_TEST_SAMPLE_RATIO, seed=999)
    
    # 2. HQ data (48 × 4 = 192 samples with augmentation)
    hq_val_base_ds = ForgeryDataset(
        config.HQ_AUTHENTIC_DIR,
        config.HQ_FORGED_DIR,
        config.HQ_MASK_DIR,
        get_standard_augmentation(config),  # Light aug for validation
        config
    )
    hq_val_ds = RepeatedDataset(hq_val_base_ds, config.VAL_HQ_MULTIPLIER, seed=999)
    
    # Combine validation datasets
    combined_val_ds = ConcatDataset([test_val_ds, hq_val_ds])
    
    if rank == 0:
        print(f"\n📊 Fine-tuning Validation Data:")
        print(f"   Test: {len(test_val_ds)} samples ({config.VAL_TEST_SAMPLE_RATIO*100}%)")
        print(f"   HQ:   {len(hq_val_ds)} samples (48 × {config.VAL_HQ_MULTIPLIER})")
        print(f"   TOTAL: {len(combined_val_ds)} samples")
        print(f"   Target distribution: {config.VAL_AUTHENTIC_RATIO*100}% Auth / {config.VAL_FORGED_RATIO*100}% Forged")
    
    # Create distributed samplers
    train_sampler = DistributedSampler(
        combined_train_ds, 
        num_replicas=world_size,
        rank=rank, 
        shuffle=True,
        seed=42
    )
    
    val_sampler = DistributedSampler(
        combined_val_ds,
        num_replicas=world_size,
        rank=rank,
        shuffle=False,
        seed=999
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        combined_train_ds,
        batch_size=config.BATCH_SIZE,
        sampler=train_sampler,
        num_workers=4,
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        combined_val_ds,
        batch_size=config.BATCH_SIZE,
        sampler=val_sampler,
        num_workers=4,
        pin_memory=True
    )
    
    return train_loader, val_loader, train_sampler, val_sampler

Writing finetune_dataset.py


In [16]:
%%writefile finetune_train.py
"""Fine-tuning Script for Final Submission"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import numpy as np
from pathlib import Path
import cv2

from finetune_config import FinetuneConfig
from finetune_dataset import create_finetune_dataloaders
from model import create_model
from losses import CombinedLoss
from clustering import cluster_embeddings, generate_masks_from_clusters
from utils import oF1_score
import warnings
warnings.filterwarnings("ignore", message="'force_all_finite' was renamed to 'ensure_all_finite'")

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12358'
    dist.init_process_group(FinetuneConfig.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

def load_pretrained_model(config, device):
    """Load best model from initial training phase"""
    if not os.path.exists(config.PRETRAINED_CHECKPOINT):
        raise FileNotFoundError(f"Pretrained checkpoint not found: {config.PRETRAINED_CHECKPOINT}")
    
    if device == 0:
        print(f"📥 Loading pretrained model from: {config.PRETRAINED_CHECKPOINT}")
    
    model = create_model(config).to(device)
    ckpt = torch.load(config.PRETRAINED_CHECKPOINT, map_location=f'cuda:{device}', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    
    # Unfreeze everything for fine-tuning
    for param in model.parameters():
        param.requires_grad = True
    
    if device == 0:
        print("✅ Pretrained model loaded. All layers unfrozen for fine-tuning.")
    
    return model

def save_checkpoint(model, optimizer, scheduler, scaler, epoch, best_of1, 
                   checkpoint_dir, is_best=False):
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.module.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_of1': best_of1
    }
    torch.save(checkpoint, os.path.join(checkpoint_dir, 'finetune_current.pth'))
    if is_best:
        torch.save(checkpoint, os.path.join(checkpoint_dir, 'finetune_best.pth'))

def compute_of1_with_distribution_filter(model, dataloader, config, device, rank):
    """
    Compute OF1 with filtering to match 30% auth / 70% forged distribution
    """
    model.eval()
    
    authentic_scores, forged_scores = [], []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validation", disable=(rank != 0))
        
        for batch in pbar:
            inputs = batch['input'].to(device)
            masks_33 = batch['masks'].cpu().numpy()
            is_forged_batch = batch['is_forged'].cpu().numpy()
            
            embeddings = model(inputs)
            embeddings_np = embeddings.cpu().numpy()
            
            for b in range(inputs.shape[0]):
                is_forged = is_forged_batch[b]
                
                # Apply distribution filtering
                import random
                if is_forged < 0.5:  # Authentic
                    if random.random() > config.VAL_AUTHENTIC_RATIO:
                        continue
                else:  # Forged
                    if random.random() > config.VAL_FORGED_RATIO:
                        continue
                
                embedding = embeddings_np[b]
                
                # Get ground truth masks
                gt_masks = []
                for i in range(6):
                    if masks_33[b, i].sum() > 0:
                        gt_mask = cv2.resize(masks_33[b, i],
                                           (config.EVAL_MASK_SIZE, config.EVAL_MASK_SIZE),
                                           interpolation=cv2.INTER_NEAREST)
                        gt_masks.append(gt_mask)
                
                # Cluster and predict
                cluster_labels = cluster_embeddings(embedding, config)
                pred_masks = generate_masks_from_clusters(
                    cluster_labels, config,
                    max_instances=config.MAX_INSTANCES,
                    embeddings=embedding
                )
                
                # Compute OF1
                if not gt_masks:
                    score = 1.0 if not pred_masks else 0.0
                else:
                    score = 0.0 if not pred_masks else oF1_score(pred_masks, gt_masks)
                
                if np.isnan(score):
                    score = 0.0
                
                if is_forged >= 0.5:
                    forged_scores.append(score)
                else:
                    authentic_scores.append(score)
            
            del inputs, embeddings, embeddings_np
            torch.cuda.empty_cache()
    
    model.train()
    
    overall = np.mean(authentic_scores + forged_scores) if (authentic_scores + forged_scores) else 0.0
    auth_of1 = np.mean(authentic_scores) if authentic_scores else 0.0
    forg_of1 = np.mean(forged_scores) if forged_scores else 0.0
    
    if rank == 0:
        print(f"📊 Validation: {len(authentic_scores)} auth, {len(forged_scores)} forged")
    
    return overall, auth_of1, forg_of1

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, 
                   epoch, config, device, rank):
    model.train()
    total_loss = 0.0
    loss_sum = {'uniformity': 0.0, 'separation': 0.0, 'triplet': 0.0}
    
    pbar = tqdm(loader, desc=f"Finetune Epoch {epoch}", disable=(rank != 0))
    for i, batch in enumerate(pbar):
        inputs = batch['input'].to(device)
        masks = batch['masks'].to(device)
        is_forged = batch['is_forged'].to(device)
        
        with autocast(enabled=config.USE_AMP):
            embeddings = model(inputs)
            loss, loss_dict = criterion(embeddings, masks, is_forged)
            loss = loss / config.GRADIENT_ACCUMULATION_STEPS
        
        scaler.scale(loss).backward()
        
        if (i + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS
        for k in loss_dict:
            loss_sum[k] += loss_dict[k]
        
        if rank == 0:
            pbar.set_postfix({
                'loss': f"{loss.item()*config.GRADIENT_ACCUMULATION_STEPS:.4f}",
                'lr': f"{optimizer.param_groups[0]['lr']:.2e}"
            })
    
    return total_loss / len(loader), {k: v / len(loader) for k, v in loss_sum.items()}

def finetune_ddp(rank, world_size, config):
    setup_ddp(rank, world_size)
    
    if rank == 0:
        config.print_config()
        Path(config.FINETUNE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        Path(config.FINETUNE_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    
    dist.barrier()
    
    # Create dataloaders
    train_loader, val_loader, train_sampler, _ = create_finetune_dataloaders(
        config, rank, world_size
    )
    
    # Load pretrained model
    model = load_pretrained_model(config, rank)
    model = DDP(model, device_ids=[rank], find_unused_parameters=True)
    
    # Optimizer with lower learning rates
    mlp_params, dino_params = model.module.get_learnable_params()
    
    optimizer = AdamW([
        {'params': mlp_params, 'lr': config.MLP_LEARNING_RATE},
        {'params': dino_params, 'lr': config.DINO_LEARNING_RATE}
    ], weight_decay=config.WEIGHT_DECAY)
    
    criterion = CombinedLoss(config).to(rank)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=config.ETA_MIN)
    scaler = GradScaler(enabled=config.USE_AMP)
    
    best_of1 = -1.0
    patience_counter = 0
    early_stop = False
    
    if rank == 0:
        print("\n🚀 Starting fine-tuning...")
    
    for epoch in range(1, config.NUM_EPOCHS + 1):
        if early_stop:
            if rank == 0:
                print(f"\n🛑 Early stopping at epoch {epoch}")
            break
        
        train_sampler.set_epoch(epoch)
        
        # Train
        loss, loss_dict = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            scaler, epoch, config, rank, rank
        )
        
        scheduler.step()
        
        # Validate
        if epoch % config.VAL_FREQUENCY == 0 or epoch == 1 or epoch == config.NUM_EPOCHS:
            overall, auth, forg = compute_of1_with_distribution_filter(
                model, val_loader, config, rank, rank
            )
            
            if rank == 0:
                print(f"\n{'='*60}")
                print(f"Fine-tune Epoch {epoch}/{config.NUM_EPOCHS}")
                print(f"Loss: {loss:.4f}")
                print(f"OF1: Overall={overall:.4f} | Auth={auth:.4f} | Forged={forg:.4f}")
                
                # Early stopping check
                if overall > best_of1 + config.EARLY_STOPPING_MIN_DELTA:
                    best_of1 = overall
                    patience_counter = 0
                    save_checkpoint(model, optimizer, scheduler, scaler, epoch,
                                  best_of1, config.FINETUNE_CHECKPOINT_DIR, is_best=True)
                    print(f"✅ New best OF1: {best_of1:.4f}")
                else:
                    patience_counter += 1
                    print(f"⏳ No improvement. Patience: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}")
                    
                    if patience_counter >= config.EARLY_STOPPING_PATIENCE:
                        print(f"🛑 Early stopping triggered")
                        early_stop = True
                
                print(f"{'='*60}\n")
        
        # Save periodic checkpoint
        if rank == 0 and epoch % 5 == 0:
            save_checkpoint(model, optimizer, scheduler, scaler, epoch,
                          best_of1, config.FINETUNE_CHECKPOINT_DIR, is_best=False)
        
        # Broadcast early stop
        if rank == 0:
            early_stop_tensor = torch.tensor([1.0 if early_stop else 0.0], device=rank)
        else:
            early_stop_tensor = torch.tensor([0.0], device=rank)
        
        dist.broadcast(early_stop_tensor, src=0)
        early_stop = early_stop_tensor.item() > 0.5
    
    if rank == 0:
        print(f"\n🏁 Fine-tuning completed! Best OF1: {best_of1:.4f}")
        print(f"✅ Best model saved to: {config.FINETUNE_CHECKPOINT_DIR}/finetune_best.pth")
    
    cleanup_ddp()

def main():
    import torch.multiprocessing as mp
    mp.spawn(finetune_ddp, args=(FinetuneConfig.WORLD_SIZE, FinetuneConfig),
             nprocs=FinetuneConfig.WORLD_SIZE, join=True)

if __name__ == '__main__':
    main()

Writing finetune_train.py


In [17]:
%%writefile extract_embeddings.py
"""Extract embeddings on GPU, save to disk for CPU tuning"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, ConcatDataset
from torch.utils.data.distributed import DistributedSampler
import numpy as np
from tqdm import tqdm
import random
import cv2
import pickle

from config import Config
from model import create_model
from dataset import ForgeryDataset, get_val_transform
from finetune_dataset import RepeatedDataset, SubsampledDataset, get_standard_augmentation

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12359'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

def load_model(config, device):
    path = os.path.join(config.CHECKPOINT_DIR, 'best_checkpoint.pth')
    if not os.path.exists(path):
        path = os.path.join(config.CHECKPOINT_DIR, 'current_checkpoint.pth')
    
    ckpt = torch.load(path, map_location=f'cuda:{device}', weights_only=False)
    model = create_model(config).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def create_tuning_dataset(config, rank):
    if rank == 0:
        print(f"\n📦 Creating Tuning Dataset...")
    
    # 1. HQ Data (48 × 4 = 192)
    hq_base_ds = ForgeryDataset(
        config.HQ_AUTHENTIC_DIR,
        config.HQ_FORGED_DIR,
        config.HQ_MASK_DIR,
        get_standard_augmentation(config),
        config
    )
    hq_repeated_ds = RepeatedDataset(hq_base_ds, 4, seed=999)
    
    # 2. Test Data (10%)
    test_base_ds = ForgeryDataset(
        config.TEST_AUTHENTIC_DIR,
        config.TEST_FORGED_DIR,
        config.TEST_MASK_DIR,
        None,
        config
    )
    test_subsample_ds = SubsampledDataset(test_base_ds, 0.10, seed=999)
    
    combined_ds = ConcatDataset([hq_repeated_ds, test_subsample_ds])
    
    if rank == 0:
        print(f"   HQ:   {len(hq_repeated_ds)} samples")
        print(f"   Test: {len(test_subsample_ds)} samples")
        print(f"   TOTAL: {len(combined_ds)} samples")
    
    return combined_ds

def extract_embeddings_ddp(rank, world_size):
    setup_ddp(rank, world_size)
    
    ds = create_tuning_dataset(Config, rank)
    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=False)
    loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=4)
    
    model = load_model(Config, rank)
    model = DDP(model, device_ids=[rank])
    
    if rank == 0:
        print(f"\n🚀 Extracting embeddings on GPU...")
    
    all_data = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting", disable=(rank != 0)):
            inputs = batch['input'].to(rank)
            masks_33 = batch['masks'].cpu().numpy()
            is_forged_batch = batch['is_forged'].cpu().numpy()
            
            embeddings = model(inputs)
            embeddings_np = embeddings.cpu().numpy()
            
            for b in range(inputs.shape[0]):
                is_forged = is_forged_batch[b]
                
                # Apply 30/70 filter
                if is_forged < 0.5:
                    if random.random() > Config.TUNING_AUTHENTIC_RATIO:
                        continue
                else:
                    if random.random() > Config.TUNING_FORGED_RATIO:
                        continue
                
                gt_masks = []
                for i in range(6):
                    if masks_33[b, i].sum() > 0:
                        gt_mask = cv2.resize(masks_33[b, i],
                                           (Config.EVAL_MASK_SIZE, Config.EVAL_MASK_SIZE),
                                           interpolation=cv2.INTER_NEAREST)
                        gt_masks.append(gt_mask)
                
                all_data.append({
                    'embeddings': embeddings_np[b],
                    'gt_masks': gt_masks
                })
            
            del inputs, embeddings, embeddings_np
            torch.cuda.empty_cache()
    
    # Gather from all ranks
    gathered = [None for _ in range(world_size)]
    dist.all_gather_object(gathered, all_data)
    
    if rank == 0:
        # Combine all data
        final_data = [item for sublist in gathered for item in sublist]
        
        print(f"\n💾 Saving {len(final_data)} embeddings to disk...")
        
        # Save to pickle
        with open('/kaggle/working/tuning_embeddings.pkl', 'wb') as f:
            pickle.dump(final_data, f)
        
        print(f"✅ Embeddings saved to: /kaggle/working/tuning_embeddings.pkl")
        print(f"   Size: {len(final_data)} samples")
    
    cleanup_ddp()

if __name__ == '__main__':
    import torch.multiprocessing as mp
    mp.spawn(extract_embeddings_ddp, args=(Config.WORLD_SIZE,), 
             nprocs=Config.WORLD_SIZE, join=True)

Writing extract_embeddings.py


In [18]:
# !python extract_embeddings.py

In [19]:
!python finetune_train.py

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
FINE-TUNING CONFIGURATION
Training Data Composition:
  - Test Data: 100.0% (5k images)
  - HQ Data: 48 × 6 = 288 samples/epoch
  - Train Data: 30.0% (~7.2k images)
  - Total: ~12488 samples/epoch

Validation Data Composition:
  - Test Data: 10.0% (500 images)
  - HQ Data: 48 × 4 = 192 samples
  - Distribution: 30.0% Auth / 70.0% Forged

Learning Rates:
  - MLP: 1e-06
  - DINOv2: 1e-06

Training:
  - Epochs: 30
  - Validate every: 3 epochs
/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:4807: UserWarning: No device id is provided via `init_process_group` or `barrier `. Using the current device set by the user. 
  warnings.warn(  # warn only once
/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:4807: UserWarning: No device id is provided via `init_process_g

In [20]:
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .
# .